<a href="https://colab.research.google.com/github/NAW0510/RepoISLAB/blob/main/Submission%20Kelompok%2012.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproduksi SapBERT-XLMR untuk Entity Linking Biomedis Lintas Bahasa

### Internship Lab KC 2026, Checkpoint 1

**Judul Project / Research:** Reproduksi *Learning Domain-Specialised Representations for Cross-Lingual Biomedical Entity Linking* (Liu dkk., ACL 2021)  
**Nama Mentee / Tim:** Kelompok 12, I Gusti Ngurah Anantawijaya Mahardika dan Jason Enkristo  
**Mentor:** Kak Agung  

Notebook ini menggabungkan dua tahap eksperimen yang dijalankan pada dua sesi Google Colab terpisah. Pembagian sel per sesi dijelaskan di Bab 4.

## Daftar Isi

1. [Abstrak / Ringkasan](#bab-1)  
2. [Pendahuluan dan Pertanyaan Penelitian](#bab-2)  
3. [Penelitian Terkait dan Rancangan Alur Penelitian](#bab-3)  
4. [Dependensi dan Reproduksibilitas](#bab-4)  
5. [Data dan Exploratory Data Analysis](#bab-5)  
6. [Preprocessing dan Feature Engineering](#bab-6)  
7. [Baseline](#bab-7)  
8. [Metode / Pendekatan yang Digunakan](#bab-8)  
9. [Validasi dan Evaluasi](#bab-9)  
10. [Hasil Eksperimen dan Analisis](#bab-10)  
11. [Interpretabilitas / Error Analysis](#bab-11)  
12. [Biaya Komputasi](#bab-12)  
13. [Model Final / Inferensi](#bab-13)  
14. [Pembahasan](#bab-14)  
15. [Keterbatasan](#bab-15)  
16. [Kesimpulan](#bab-16)  
17. [Referensi](#bab-17)

<a id="bab-1"></a>
## 1. Abstrak / Ringkasan

Kami mereproduksi model XLMR + SAP dari Liu dkk. [1] untuk *biomedical entity linking* lintas bahasa, yaitu memetakan frasa medis dari berbagai bahasa ke konsep UMLS (CUI). Pada Tahap 1, XLM-R base dilatih ulang dengan 21.082.913 pasangan sinonim UMLS 2020AA dari semua bahasa memakai kode resmi penulis. Pada Tahap 2, model tersebut dilatih lanjut dengan data terjemahan umum Inggris–Thai dari kamus MUSE dan pasangan judul Wikipedia (WikiTitles) yang kami rekonstruksi dari Wikidata. Evaluasi memakai benchmark XL-BEL dengan metrik P@1 dan P@5 pada 1.000 frasa uji per bahasa.

Tahap 1 menghasilkan P@1/P@5 sebesar 21,10/26,90 untuk Thai dan 18,00/22,40 untuk Mandarin (paper: 20,6/27,5 dan 18,8/23,0). Pada Tahap 2, P@1 Thai naik menjadi 25,50 dengan MUSE, 29,70 dengan WikiTitles, dan 30,40 dengan gabungan keduanya, dengan urutan yang sama seperti Tabel 3 paper. Semua selisih dengan paper berada dalam 0,8 poin, lebih kecil dari interval kepercayaan 95% sekitar ±2,5 poin untuk 1.000 frasa uji. Hasil ini menjadi dasar tahap berikutnya, yaitu adaptasi model ke bahasa Indonesia.

<a id="bab-2"></a>
## 2. Pendahuluan dan Pertanyaan Penelitian

Satu konsep medis dapat ditulis dengan banyak cara dan dalam banyak bahasa, misalnya "headache", "cephalgia", dan "sakit kepala". *Biomedical entity linking* memetakan setiap sebutan seperti itu ke satu konsep standar di basis pengetahuan seperti UMLS, sehingga data dari sumber dan bahasa yang berbeda dapat diolah bersama. Masalahnya, sumber daya biomedis didominasi bahasa Inggris. Banyak bahasa hanya memiliki sedikit nama di UMLS, dan bahasa Thai sama sekali tidak memilikinya.

Liu dkk. [1] menunjukkan bahwa pelatihan SAP dengan sinonim UMLS dari semua bahasa menghasilkan encoder lintas bahasa yang jauh lebih baik daripada XLM-R biasa. Mereka juga membangun benchmark XL-BEL (10 bahasa) dan menunjukkan bahwa data terjemahan umum dapat menaikkan kinerja pada bahasa yang miskin data biomedis.

Tantangan utama reproduksi ini:
- kamus target berisi 399.931 nama dengan banyak konsep yang mirip;
- bahasa uji seperti Thai tidak memiliki satu pun nama di UMLS;
- sebagian data penulis tidak tersedia lagi (tautan data terjemahan sudah tidak aktif);
- pelatihan Tahap 1 membutuhkan sekitar 82 ribu langkah, melebihi batas satu sesi Colab.

Tujuan Checkpoint 1 adalah mereproduksi temuan utama paper sebagai fondasi project. Bahasa Indonesia tidak termasuk dalam XL-BEL dan posisinya mirip Thai, sehingga temuan reproduksi ini langsung relevan untuk tahap berikutnya.

- **RQ1:** Apakah skor XLMR + SAP_all_syn (Tabel 2 paper) dapat direproduksi dengan melatih ulang dari XLM-R base memakai kode resmi?
- **RQ2:** Apakah kenaikan dari data terjemahan umum Inggris–Thai (Tabel 3 paper) dapat direproduksi, termasuk dengan WikiTitles hasil rekonstruksi?
- **RQ3:** Perbedaan apa saja antara paper dan implementasi resminya, dan seberapa besar pengaruhnya terhadap hasil?

<a id="bab-3"></a>
## 3. Penelitian Terkait dan Rancangan Alur Penelitian

### 3.1 Penelitian Terkait

- **SapBERT [2].** Melatih encoder BERT dengan *metric learning* agar nama-nama dengan CUI yang sama berdekatan di ruang vektor, memakai Multi-Similarity loss dan *online hard mining*. Tanpa data latih khusus tugas, SapBERT mencapai hasil terbaik pada enam benchmark *medical entity linking*. Metode inilah yang menjadi dasar seluruh eksperimen kami.
- **Liu dkk. [1], paper yang direproduksi.** Menerapkan SAP pada XLM-R dengan sinonim UMLS semua bahasa (`all_syn`) dan membandingkannya dengan sinonim Inggris saja (`en_syn`). Paper ini juga memperkenalkan XL-BEL dan menunjukkan bahwa pelatihan lanjut dengan data terjemahan umum (MUSE dan judul Wikipedia) menaikkan kinerja bahasa yang miskin data.
- **XLM-R [3].** Encoder Transformer multibahasa yang dilatih pada teks web 100 bahasa. Kami memakai versi base sebagai model awal.
- **Multi-Similarity loss [4].** Fungsi loss yang membobot setiap pasangan berdasarkan kemiripannya dengan pasangan positif dan negatif lain. Loss ini dipakai di kedua tahap.
- **MUSE [5].** Kumpulan kamus terjemahan kata dwibahasa dari Meta. Kamus Inggris–Thai dipakai di Tahap 2.
- **AnnoHID [6].** Membangun anotasi normalisasi konsep medis untuk teks konsultasi kesehatan berbahasa Indonesia, dengan menerjemahkan istilah ke bahasa Inggris memakai LLM sebelum mencarinya di SNOMED CT. Pendekatan berbasis terjemahan ini menjadi pembanding untuk pendekatan lintas bahasa langsung yang akan kami kembangkan setelah reproduksi.

### 3.2 Rancangan Alur Penelitian

```
Tahap 1
UMLS 2020AA (MRCONSO, semua bahasa)
  -> pasangan sinonim per CUI (21.082.913 pasangan)
  -> 10 shard acak
  -> training SAP dari XLM-R base (1 epoch)
  -> evaluasi XL-BEL Thai dan Mandarin

Tahap 2
model Tahap 1 + data paralel Inggris-Thai (MUSE, WikiTitles, gabungan)
  -> training lanjutan 5 epoch
  -> evaluasi XL-BEL Thai
  -> perbandingan dengan Tabel 2 dan Tabel 3 paper
```

### 3.3 Penjelasan Alur

1. **Data Tahap 1.** Semua nama di MRCONSO dikelompokkan per CUI. Dari setiap kelompok dibentuk pasangan sinonim berformat `CUI||nama1||nama2`.
2. **Pembagian shard.** Data dibagi menjadi 10 bagian acak supaya pelatihan dapat dilanjutkan bila sesi Colab terputus.
3. **Training Tahap 1.** XLM-R base dilatih dengan `train.py` resmi dan hyperparameter dari `train/pretrain.sh`.
4. **Data Tahap 2.** Kamus MUSE dan pasangan judul Wikipedia dikonversi ke format yang sama, dengan satu label unik per pasangan.
5. **Training Tahap 2.** Model Tahap 1 dilatih lanjut dengan tiap data paralel secara terpisah, memakai argumen `train/xling_train.sh`.
6. **Evaluasi.** Semua model dievaluasi dengan `evaluation/eval.py` resmi pada XL-BEL v0.0, lalu dibandingkan dengan angka di paper.

<a id="bab-4"></a>
## 4. Dependensi dan Reproduksibilitas

- **Python:** 3.13 (runtime Google Colab)
- **Framework:** PyTorch, `transformers` 4.45.2, `huggingface-hub` 0.36.2, `pytorch-metric-learning` 2.9.0, `wandb`, `sentencepiece`, `duckdb` (rekonstruksi WikiTitles). `transformers` versi 5 tidak dipakai karena fungsi `batch_encode_plus` yang dipakai kode Liu sudah dihapus di versi tersebut.
- **Kode acuan:** repo `cambridgeltl/sapbert`, commit `1a528f66d277d91f99b6463b43ce5309a9279506`
- **Seed:** 33 (pembentukan pasangan, pembagian shard, dan training)
- **Device:** GPU NVIDIA L4 untuk Tahap 1, GPU NVIDIA T4 untuk Tahap 2 (Colab)

**Pembagian sesi.** Seluruh notebook tidak dapat dijalankan dengan satu kali *Run All*, karena Tahap 1 membutuhkan sekitar 9,4 jam training dan kedua tahap dijalankan pada runtime yang berbeda.

| Sesi | Bab | Runtime | Isi |
|---|---|---|---|
| Tahap 1 | 4, 5, 6.1, 7, 8.1, 9.1, 13 | GPU L4 (persiapan data cukup CPU) | data UMLS, training SAP, evaluasi TH dan ZH |
| Tahap 2 | 6.2, 8.2, 9.2, 10 | GPU T4 | data paralel, training lanjutan, evaluasi TH |

**Kredensial.** `WANDB_API_KEY` dan `HF_TOKEN` disimpan di Colab Secrets. API key UTS (UMLS) dimasukkan lewat prompt saat mengunduh data dan tidak ditulis di notebook.

**Lisensi.** Data UMLS hanya boleh dipakai pemegang lisensi, sehingga file UMLS, kamus evaluasi, dan berkas `predictions_eval.json` tidak disertakan di repository. Output sel yang menampilkan isi UMLS sudah dihapus.

**Output.** Output yang ditampilkan berasal dari eksekusi asli (Tahap 1 pada 25–26 Oktober 2026). Log W&B dan progress bar dihapus dari output. Sel yang ditambahkan untuk notebook gabungan ini ditandai `# (dokumentasi)`; sel tersebut, serta sel konfigurasi yang disederhanakan, belum memiliki output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, time, shutil, subprocess, getpass

REPO = Path('/content/reproduce_liu')
STORE = Path('/content/drive/MyDrive/sapbert_liu')
for folder in ('models', 'results', 'runs', 'data'):
    (STORE / folder).mkdir(parents=True, exist_ok=True)

In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/cambridgeltl/sapbert.git', str(REPO)], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Runtime belum memakai GPU.")

print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

%pip uninstall -q -y diffusers gradio gradio-client

%pip install -q "transformers==4.45.2" "huggingface-hub==0.36.2" "pytorch-metric-learning>=2,<3" "wandb>=0.17" "sentencepiece" "scikit-learn" "pandas"

import transformers
import huggingface_hub
import wandb
import pytorch_metric_learning
from transformers import AutoTokenizer, AutoModel

print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("W&B:", wandb.__version__)
print("Setup library berhasil!")

Torch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA L4
Mon Oct 26 00:50:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             14W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-

In [ ]:
def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

import wandb
wandb_key = get_secret('WANDB_API_KEY')
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key (Enter untuk melewati W&B): ').strip()
LOG_TO_WANDB = bool(wandb_key)
if LOG_TO_WANDB:
    wandb.login(key=wandb_key)
    os.environ['WANDB_MODE'] = 'online'
    print('W&B login berhasil.')
else:
    print('W&B dilewati; log/hasil tetap disimpan ke Google Drive.')
wandb_key = None
WANDB_PROJECT = 'sapbert'

W&B login berhasil.


<a id="bab-5"></a>
## 5. Data dan Exploratory Data Analysis

### 5.1 Deskripsi Dataset

| Dataset | Peran | Ukuran | Sumber dan lisensi |
|---|---|---|---|
| UMLS 2020AA, tabel MRCONSO | data latih Tahap 1 | 15.479.756 nama, 4.281.184 CUI | NLM, lisensi UMLS (tidak disertakan) |
| XL-BEL v0.0 | data uji | 1.000 frasa per bahasa (TH, ZH, DE dipakai di sini) | repo Liu dkk., dibangun dari tautan Wikipedia |
| Kamus `wikimed_only` | kamus target saat evaluasi | 399.931 nama | repo Liu dkk., berisi nama UMLS |
| MUSE en–th | data latih Tahap 2 | 24.658 pasangan | Meta, repo `facebookresearch/MUSE` |
| WikiTitles en–th | data latih Tahap 2 | 147.100 pasangan | `wikimedia/wikidata-title-desc` di Hugging Face, snapshot 2026-03-30 |

Tugas ini bukan klasifikasi. Setiap frasa uji dipetakan ke CUI dengan mencari nama terdekat di kamus. Tidak ada pembagian train/validation dari XL-BEL: data latih berasal dari UMLS dan data paralel, sedangkan XL-BEL hanya dipakai sebagai data uji. Hyperparameter mengikuti paper dan skrip resmi, sehingga data uji tidak pernah dipakai untuk memilih model.

In [ ]:
from google.colab import drive
from getpass import getpass
from pathlib import Path
import subprocess
import zipfile

drive.mount("/content/drive")

folder = Path("/content/drive/MyDrive/sapbert_liu/data/2020AA")
folder.mkdir(parents=True, exist_ok=True)

filename = "umls-2020AA-mrconso.zip"
target = folder / filename
partial = folder / (filename + ".part")

source = (
    "https://download.nlm.nih.gov/umls/kss/"
    "2020AA/umls-2020AA-mrconso.zip"
)

if target.exists() and zipfile.is_zipfile(target):
    print("Dataset sudah ada di Drive. Tidak perlu download ulang.")
else:
    key = getpass("Masukkan UMLS API Key: ").strip()
    if not key:
        raise ValueError("API key tidak boleh kosong.")

    print("Mengunduh UMLS 2020AA...")

    cmd = [
        "curl", "-fL",
        "--retry", "8",
        "--retry-all-errors",
        "--retry-delay", "10",
        "--connect-timeout", "30",
        "-C", "-",
        "-G",
        "--data-urlencode", f"url={source}",
        "--data-urlencode", f"apiKey={key}",
        "-o", str(partial),
        "https://uts-ws.nlm.nih.gov/download"
    ]

    result = subprocess.run(cmd)

    if result.returncode != 0:
        raise RuntimeError(
            "Download belum selesai. Periksa API key/koneksi. "
            "Jika server mendukung resume, jalankan cell lagi."
        )

    if not zipfile.is_zipfile(partial):
        raise RuntimeError(
            "File bukan ZIP valid. Periksa API key dan respons server."
        )

    partial.rename(target)
    print("DOWNLOAD BERHASIL!")

if target.exists() and zipfile.is_zipfile(target):
    print("Lokasi:", target)
    print("Ukuran:", round(target.stat().st_size / 1024**2, 1), "MB")
    with zipfile.ZipFile(target) as z:
        print("Contoh isi ZIP:", z.namelist()[:10])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Masukkan UMLS API Key: ··········
Mengunduh UMLS 2020AA...
DOWNLOAD BERHASIL!
Lokasi: /content/drive/MyDrive/sapbert_liu/data/2020AA/umls-2020AA-mrconso.zip
Ukuran: 406.6 MB
Contoh isi ZIP: ['MRCONSO.RRF']


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/sapbert_liu/data")
ZIP = BASE / "2020AA/umls-2020AA-mrconso.zip"
assert ZIP.exists(), "ZIP MRCONSO tidak ditemukan."

with zipfile.ZipFile(ZIP) as z:
    info = z.getinfo("MRCONSO.RRF")
    print("Ukuran MRCONSO.RRF:", round(info.file_size / 1024**3, 2), "GB")
    with z.open("MRCONSO.RRF") as f:
        kolom = f.readline().decode("utf-8").split("|")
print("Jumlah kolom per baris:", len(kolom))

### 5.2 Distribusi Data Uji

Data uji XL-BEL berformat `CUI||frasa`. Sebagian frasa memiliki lebih dari satu CUI yang dianggap benar, ditulis dengan pemisah `|`. Sel berikut menghitung ringkasan data uji untuk bahasa yang dipakai, tanpa menampilkan isi kamus.

In [ ]:
import pandas as pd

XLBEL_DIR = REPO / 'evaluation/xl_bel'
ringkasan = []
for lang in ('th', 'zh', 'de'):
    lines = (XLBEL_DIR / f'xlbel_v0.0/{lang}_1k_test_query.txt').read_text(encoding='utf-8').splitlines()
    cui_fields = [line.split('||')[0] for line in lines]
    ringkasan.append({
        'bahasa': lang,
        'jumlah frasa': len(lines),
        'CUI unik': len({c for field in cui_fields for c in field.split('|')}),
        'frasa dengan >1 CUI benar': sum('|' in field for field in cui_fields),
    })

with open(XLBEL_DIR / 'umls_onto_all_lang_cased_wikimed_only_399931.txt', encoding='utf-8') as f:
    print('Jumlah nama di kamus:', sum(1 for _ in f))
pd.DataFrame(ringkasan)

### 5.3 Kualitas Data

- **Duplikat dan format.** Saat membentuk pasangan Tahap 1, nama yang sama dalam satu CUI hanya diambil sekali, dan nama yang mengandung `||` dibuang karena bentrok dengan format data. Data Tahap 2 divalidasi di Bab 6.2: setiap baris harus berisi tiga kolom, label harus unik, dan tidak boleh ada pasangan ganda.
- **Pasangan identik di MUSE.** Sebagian besar isi kamus MUSE en–th ternyata bukan terjemahan, melainkan kata yang sama di kedua sisi (misalnya `utc` dan `utc`). Proporsinya dihitung di Bab 6.2. Pasangan ini tetap dipakai karena tanpanya jumlah data jauh di bawah 25.332 pasangan yang dilaporkan paper.
- **Kebocoran data.** XL-BEL tidak dipakai untuk training maupun pemilihan model. Satu hal yang masih terbuka: frasa uji XL-BEL berasal dari teks tautan Wikipedia, sedangkan WikiTitles berisi judul artikel Wikipedia. Tumpang tindih keduanya belum diukur.
- **Isi data uji.** Pemeriksaan sekilas data uji XL-BEL menunjukkan banyak nama organisme dan konsep non-klinis, karena data dibangun dari Wikipedia tanpa penyaringan jenis konsep.

### 5.4 Perbedaan Distribusi Data Latih dan Data Uji

Data latih Tahap 1 berupa nama-nama baku dari kosakata UMLS dalam banyak bahasa, tetapi tanpa satu pun nama Thai. Data uji berupa teks tautan dari artikel Wikipedia berbahasa Thai. Perbedaan bahasa dan bentuk teks ini adalah tantangan utama yang diukur benchmark. Tahap 2 menjembatani sebagian perbedaan itu dengan data paralel Inggris–Thai.

<a id="bab-6"></a>
## 6. Preprocessing dan Feature Engineering

### 6.1 Tahap 1: pasangan sinonim UMLS

Setiap baris MRCONSO dibaca sebagai (CUI, bahasa, nama). Nama-nama dalam satu CUI dipasangkan satu sama lain, dengan batas maksimal 50 pasangan per CUI (diambil acak dengan seed 33) untuk membatasi dominasi konsep yang memiliki sangat banyak sinonim. Nama dibiarkan apa adanya (*cased*) karena XLM-R membedakan huruf besar dan kecil. Data `en_syn` (sinonim bahasa Inggris saja) juga disiapkan untuk ablasi berikutnya, tetapi belum dilatih.

In [ ]:
import io
import random
import zipfile
from itertools import groupby, combinations
from bisect import bisect_right

OUTPUT = BASE / "umls2020aa_all_pairs.txt"
TEMP = BASE / "umls2020aa_all_pairs.txt.part"
rng = random.Random(33)

def parse_row(line):
    fields = line.rstrip("\r\n").split("|")
    return fields[0], fields[1], fields[14]

def make_pairs(names):
    n = len(names)
    total = n * (n - 1) // 2

    if total <= 50:
        yield from combinations(names, 2)
    else:
        starts = [i * (2*n-i-1)//2 for i in range(n-1)]
        for k in rng.sample(range(total), 50):
            i = bisect_right(starts, k) - 1
            j = i + 1 + k - starts[i]
            yield names[i], names[j]

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    print("Dataset sudah ada. Tidak diproses ulang.")
else:
    total_rows = 0
    total_cui = 0
    total_pairs = 0

    with zipfile.ZipFile(ZIP) as z:
        with z.open("MRCONSO.RRF") as raw:
            with io.TextIOWrapper(raw, encoding="utf-8") as f:
                rows = (parse_row(line) for line in f)

                with open(TEMP, "w", encoding="utf-8") as out:
                    for cui, group in groupby(rows, key=lambda x: x[0]):
                        seen = set()
                        names = []

                        for _, lang, name in group:
                            total_rows += 1
                            name = name.strip()

                            if name and "||" not in name and name not in seen:
                                seen.add(name)
                                names.append(name)

                        total_cui += 1

                        for a, b in make_pairs(names):
                            out.write(f"{cui}||{a}||{b}\n")
                            total_pairs += 1

                        if total_cui % 500000 == 0:
                            print(f"{total_cui:,} konsep diproses")

    TEMP.replace(OUTPUT)

    print("\nPREPROCESSING BERHASIL!")
    print("Baris MRCONSO:", f"{total_rows:,}")
    print("Jumlah CUI:", f"{total_cui:,}")
    print("Pasangan sinonim:", f"{total_pairs:,}")
    print("File output:", OUTPUT)
    print("Ukuran GB:", round(OUTPUT.stat().st_size / 1024**3, 2))

500,000 konsep diproses
1,000,000 konsep diproses
1,500,000 konsep diproses
2,000,000 konsep diproses
2,500,000 konsep diproses
3,000,000 konsep diproses
3,500,000 konsep diproses
4,000,000 konsep diproses

PREPROCESSING BERHASIL!
Baris MRCONSO: 15,479,756
Jumlah CUI: 4,281,184
Pasangan sinonim: 21,082,913
File output: /content/drive/MyDrive/sapbert_liu/data/umls2020aa_all_pairs.txt
Ukuran GB: 1.99


In [ ]:
import io
import json
import random
import zipfile
from itertools import groupby, combinations
from bisect import bisect_right
from pathlib import Path

DATA = Path("/content/drive/MyDrive/sapbert_liu/data")
ZIP = DATA / "2020AA/umls-2020AA-mrconso.zip"
OUTPUT = DATA / "umls2020aa_en_pairs.txt"
TEMP = DATA / "umls2020aa_en_pairs.txt.part"
STATS = OUTPUT.with_suffix(".stats.json")
rng = random.Random(33)

assert ZIP.exists(), "ZIP MRCONSO tidak ditemukan; unduh terlebih dahulu (Bab 5.1)."

def parse_row(line):
    fields = line.rstrip("\r\n").split("|")
    return fields[0], fields[1], fields[14]

def make_pairs(names):
    n = len(names)
    total = n * (n - 1) // 2

    if total <= 50:
        yield from combinations(names, 2)
    else:
        starts = [i * (2*n-i-1)//2 for i in range(n-1)]
        for k in rng.sample(range(total), 50):
            i = bisect_right(starts, k) - 1
            j = i + 1 + k - starts[i]
            yield names[i], names[j]

if OUTPUT.exists() and STATS.exists():
    print("Dataset en_syn sudah ada. Tidak diproses ulang.")
else:
    total_rows = 0
    eng_rows = 0
    total_cui = 0
    total_pairs = 0

    with zipfile.ZipFile(ZIP) as z:
        with z.open("MRCONSO.RRF") as raw:
            with io.TextIOWrapper(raw, encoding="utf-8") as f:
                rows = (parse_row(line) for line in f)

                with open(TEMP, "w", encoding="utf-8") as out:
                    for cui, group in groupby(rows, key=lambda x: x[0]):
                        seen = set()
                        names = []

                        for _, lang, name in group:
                            total_rows += 1
                            if lang != "ENG":
                                continue
                            eng_rows += 1
                            name = name.strip()

                            if name and "||" not in name and name not in seen:
                                seen.add(name)
                                names.append(name)

                        if not names:
                            continue
                        total_cui += 1

                        for a, b in make_pairs(names):
                            out.write(f"{cui}||{a}||{b}\n")
                            total_pairs += 1

                        if total_cui % 500000 == 0:
                            print(f"{total_cui:,} konsep berbahasa Inggris diproses")

    TEMP.replace(OUTPUT)
    STATS.write_text(json.dumps({
        "variant": "en_syn", "mrconso_rows": total_rows, "eng_rows": eng_rows,
        "cui_with_eng_name": total_cui, "pairs": total_pairs,
    }, indent=2), encoding="utf-8")

    print("\nPREPROCESSING en_syn BERHASIL!")

stats = json.loads(STATS.read_text())
print("Baris MRCONSO:", f"{stats['mrconso_rows']:,}")
print("Baris berbahasa Inggris:", f"{stats['eng_rows']:,}")
print("CUI dengan nama Inggris:", f"{stats['cui_with_eng_name']:,}")
print("Pasangan sinonim:", f"{stats['pairs']:,}")
print("File output:", OUTPUT)
print("Ukuran GB:", round(OUTPUT.stat().st_size / 1024**3, 2))

500,000 konsep berbahasa Inggris diproses
1,000,000 konsep berbahasa Inggris diproses
1,500,000 konsep berbahasa Inggris diproses
2,000,000 konsep berbahasa Inggris diproses
2,500,000 konsep berbahasa Inggris diproses
3,000,000 konsep berbahasa Inggris diproses
3,500,000 konsep berbahasa Inggris diproses
4,000,000 konsep berbahasa Inggris diproses

PREPROCESSING en_syn BERHASIL!
Baris MRCONSO: 15,479,756
Baris berbahasa Inggris: 10,953,636
CUI dengan nama Inggris: 4,280,540
Pasangan sinonim: 11,530,468
File output: /content/drive/MyDrive/sapbert_liu/data/umls2020aa_en_pairs.txt
Ukuran GB: 0.97


In [ ]:
from huggingface_hub import snapshot_download

MODEL = STORE / "models/xlmr_base"

ready = (
    (MODEL / "config.json").exists()
    and (
        list(MODEL.glob("*.safetensors"))
        or list(MODEL.glob("*.bin"))
    )
)

if ready:
    print("XLM-R Base sudah tersedia di Drive.")
else:
    snapshot_download(
        repo_id="FacebookAI/xlm-roberta-base",
        local_dir=str(MODEL),
        allow_patterns=["*.json", "*.model", "*.safetensors"]
    )
    print("Model berhasil disimpan!")

print("Lokasi:", MODEL)

Model berhasil disimpan!
Lokasi: /content/drive/MyDrive/sapbert_liu/models/xlmr_base


In [ ]:

from pathlib import Path

VARIANT = "all_syn"

STORE = Path("/content/drive/MyDrive/sapbert_liu")
VARIANTS = {
    "all_syn": {
        "pairs": STORE / "data/umls2020aa_all_pairs.txt",
        "expected_pairs": 21_082_913,
        "manifest": STORE / "data/sap_full_shards_manifest.json",
        "shards": Path("/content/sap_full_shards"),
        "run": STORE / "runs/sapbert_full_umls2020aa",
        "hf_repo": "sapbert-full-umls2020aa-private",
        "result_prefix": "sapfull",
        "data_note": "UMLS 2020AA MRCONSO semua bahasa, maks 50 pasangan per CUI",
    },
    "en_syn": {
        "pairs": STORE / "data/umls2020aa_en_pairs.txt",
        "expected_pairs": None,
        "manifest": STORE / "data/sap_en_syn_shards_manifest.json",
        "shards": Path("/content/sap_en_syn_shards"),
        "run": STORE / "runs/sapbert_en_syn_umls2020aa",
        "hf_repo": "sapbert-en-syn-umls2020aa-private",
        "result_prefix": "sap_en_syn",
        "data_note": "UMLS 2020AA MRCONSO hanya LAT=ENG, maks 50 pasangan per CUI",
    },
}
assert VARIANT in VARIANTS, f"VARIANT harus salah satu dari {list(VARIANTS)}"
V = VARIANTS[VARIANT]
print("Varian aktif:", VARIANT)
print("Data  :", V["pairs"])
print("Output:", V["run"])

Varian aktif: all_syn
Data  : /content/drive/MyDrive/sapbert_liu/data/umls2020aa_all_pairs.txt
Output: /content/drive/MyDrive/sapbert_liu/runs/sapbert_full_umls2020aa


In [ ]:

from pathlib import Path
import json
import random

assert "V" in globals(), "Pilih varian di Bab 6.1 terlebih dahulu."
SOURCE = V["pairs"]
SHARD_DIR = V["shards"]
MANIFEST = V["manifest"]
N_SHARDS = 10
SHARD_SEED = 33

assert SOURCE.is_file(), f"Dataset {VARIANT} tidak ditemukan: {SOURCE}. Buat pasangan sinonimnya terlebih dahulu (Bab 6.1)."
EXPECTED_PAIRS = V["expected_pairs"]
if EXPECTED_PAIRS is None:
    stats_file = SOURCE.with_suffix(".stats.json")
    assert stats_file.is_file(), f"{stats_file.name} belum ada; buat data {VARIANT} terlebih dahulu (Bab 6.1)."
    EXPECTED_PAIRS = json.loads(stats_file.read_text())["pairs"]

def shard_path(k):
    return SHARD_DIR / f"shard_{k:02d}.txt"

def count_lines(path):
    with path.open("rb") as f:
        return sum(1 for _ in f)

manifest = json.loads(MANIFEST.read_text()) if MANIFEST.is_file() else None
if manifest:
    assert manifest["n_shards"] == N_SHARDS and manifest["seed"] == SHARD_SEED, (
        "N_SHARDS/SHARD_SEED berbeda dari manifest. Kembalikan nilainya agar training bisa dilanjutkan.")

ready = manifest is not None and all(
    shard_path(k).is_file() and count_lines(shard_path(k)) == manifest["counts"][k]
    for k in range(N_SHARDS)
)

if ready:
    print(f"Shard {VARIANT} sudah ada di runtime ini. Tidak dibuat ulang.")
else:
    print(f"Membuat shard {VARIANT} dari", SOURCE, "(beberapa menit) ...")
    SHARD_DIR.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SHARD_SEED)
    counts = [0] * N_SHARDS
    outs = [shard_path(k).open("w", encoding="utf-8") for k in range(N_SHARDS)]
    try:
        with SOURCE.open(encoding="utf-8") as src:
            for line in src:

                if line.count("||") != 2:
                    raise ValueError(f"Baris tidak berformat CUI||nama1||nama2: {line[:120]!r}")
                k = rng.randrange(N_SHARDS)
                outs[k].write(line)
                counts[k] += 1
    finally:
        for f in outs:
            f.close()
    assert sum(counts) == EXPECTED_PAIRS, f"Jumlah pasangan {sum(counts):,}, seharusnya {EXPECTED_PAIRS:,}"
    if manifest:
        assert counts == manifest["counts"], "Shard berbeda dari sesi sebelumnya. Jangan lanjutkan training."
    else:
        MANIFEST.write_text(json.dumps({
            "variant": VARIANT, "source": str(SOURCE), "total_pairs": sum(counts),
            "n_shards": N_SHARDS, "seed": SHARD_SEED, "counts": counts,
        }, indent=2), encoding="utf-8")
    print("Shard berhasil dibuat!")

counts = json.loads(MANIFEST.read_text())["counts"]
print("Varian:", VARIANT, "| lokasi:", SHARD_DIR)
print("Pasangan per shard:", [f"{c:,}" for c in counts])
print("Total:", f"{sum(counts):,}")

Membuat shard all_syn dari /content/drive/MyDrive/sapbert_liu/data/umls2020aa_all_pairs.txt (beberapa menit) ...
Shard berhasil dibuat!
Varian: all_syn | lokasi: /content/sap_full_shards
Pasangan per shard: ['2,105,815', '2,108,710', '2,109,260', '2,107,283', '2,107,949', '2,108,028', '2,108,470', '2,108,770', '2,107,992', '2,110,636']
Total: 21,082,913


### 6.2 Tahap 2: data paralel Inggris–Thai

Tahap 2 dijalankan pada sesi Colab terpisah. Tiga sel pertama menyiapkan konfigurasi sesi tersebut.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, time, re, csv, math, random

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/sapbert_liu/experiment2_th')
INPUT = ROOT / 'input'
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
RESULTS = ROOT / 'results'
REPORTS = ROOT / 'reports'
for p in (INPUT, DATA, RUNS, RESULTS, REPORTS): p.mkdir(parents=True, exist_ok=True)

MY_STAGE1 = Path('/content/drive/MyDrive/sapbert_liu/runs/sapbert_full_umls2020aa/final')
MY_MUSE_PAIRS = Path('/content/drive/MyDrive/sapbert_liu/data/muse/en_th_muse_pairs.txt')

REPO = Path('/content/reproduce_liu')
LOCAL_STAGE1 = Path('/content/sapbert_stage1_cache')
LOCAL_OUT = Path('/content/exp2_out')
WANDB_PROJECT = 'sapbert-liu-exp2-th'
WANDB_GROUP = '2021-table3-th'

SEED = 33
LR = 2e-5
EPOCHS = 5
MAX_LEN = 25
BATCH_SIZE = 256
MINER_MARGIN = 0.2
LOSS_PARAMS = 'kode'
SAVE_EPOCH_CHECKPOINTS = True
assert LOSS_PARAMS in ('kode', 'paper')

def run_name(variant):
    """Nama run; ablasi parameter loss paper diberi akhiran _paperloss."""
    return variant if LOSS_PARAMS == 'kode' else f'{variant}_paperloss'

print('Drive eksperimen  :', ROOT)
print('Model Eksperimen 1:', MY_STAGE1)
print('Batch / loss      :', BATCH_SIZE, '/', LOSS_PARAMS)

Mounted at /content/drive
Drive eksperimen  : /content/drive/MyDrive/sapbert_liu/experiment2_th
Model Eksperimen 1: /content/drive/MyDrive/sapbert_liu/runs/sapbert_full_umls2020aa/final
Batch / loss      : 256 / kode


In [ ]:
import importlib.metadata

def ver(pkg):
    try: return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError: return None

if not (ver('transformers') == '4.45.2' and ver('huggingface-hub') == '0.36.2' and ver('pytorch-metric-learning') and ver('wandb')):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.45.2', 'huggingface-hub==0.36.2',
        'pytorch-metric-learning>=2,<3', 'wandb>=0.17', 'sentencepiece'], check=True)
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/cambridgeltl/sapbert.git', str(REPO)],check=True)
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('Commit SapBERT:', COMMIT)
print('Transformers:', ver('transformers'), 'PML:', ver('pytorch-metric-learning'))

WRAPPER = REPO/'src/model_wrapper.py'
s = WRAPPER.read_text()
pattern = r'(def embed_dense\([^\n]*?batch_size=)(\d+)'
match = re.search(pattern, s)
if not match:
    raise RuntimeError('Signature embed_dense berubah; periksa sumber sebelum evaluasi.')
if int(match.group(2)) > 64:
    WRAPPER.write_text(re.sub(pattern, lambda m: m.group(1)+'64', s, count=1))

Commit SapBERT: 1a528f66d277d91f99b6463b43ce5309a9279506
Transformers: 4.45.2 PML: 2.9.0


In [ ]:
try:
    from google.colab import userdata
    WB_KEY = userdata.get('WANDB_API_KEY')
except Exception:
    WB_KEY = os.environ.get('WANDB_API_KEY')
if WB_KEY:
    os.environ['WANDB_API_KEY'] = WB_KEY
    os.environ['WANDB_MODE'] = 'online'
    print('W&B aktif:', WANDB_PROJECT)
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B belum dikoneksi. Di Colab Secrets tambahkan WANDB_API_KEY untuk logging otomatis.')

W&B aktif: sapbert-liu-exp2-th


**WikiTitles.** Paper memakai 107.398 pasangan judul artikel Wikipedia Inggris–Thai, tetapi tautan data penulis sudah tidak aktif. Kami merekonstruksinya dari dataset `wikimedia/wikidata-title-desc` (snapshot 2026-03-30): judul artikel bahasa Inggris dan Thai yang terhubung ke QID Wikidata yang sama dipasangkan, lalu halaman non-artikel (Category, Template, File, dan sejenisnya) dibuang. Hasilnya 147.100 pasangan. Perbedaan jumlah ini dicatat sebagai perbedaan kondisi reproduksi.

In [ ]:
%pip install -q "duckdb>=1.1,<2" "huggingface_hub>=0.34,<1" "hf_xet>=1,<2"

In [ ]:
import json, hashlib, os, re
from pathlib import Path

SOURCE = ROOT / 'data' / 'source'
SOURCE.mkdir(parents=True, exist_ok=True)
RAW = Path('/content/wikimedia_wikidata_20260330_parquet')
OUT = SOURCE / 'wt_en_th_reconstructed_20260330.txt'
META = SOURCE / 'wt_en_th_reconstructed_20260330.metadata.json'
REPO_ID = 'wikimedia/wikidata-title-desc'
REV = 'snapshot-2026-03-30'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def wt_valid():
    """True jika hasil rekonstruksi di Drive lengkap dan checksum-nya cocok."""
    if not (OUT.exists() and META.exists()):
        return False
    saved = json.loads(META.read_text(encoding='utf-8'))
    return saved.get('revision') == REV and saved.get('sha256') == sha256(OUT)

print('Output permanen:', OUT)
print('Sudah ada dan valid:', wt_valid())

Output permanen: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/source/wt_en_th_reconstructed_20260330.txt
Sudah ada dan valid: False


In [ ]:
if wt_valid():
    print('Data WikiTitles sudah ada di Drive; unduhan parquet dilewati.')
else:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    repo_files = api.list_repo_files(REPO_ID, repo_type='dataset', revision=REV)
    files_en = sorted(x for x in repo_files if x.startswith('data/en/') and x.endswith('.parquet'))
    files_th = sorted(x for x in repo_files if x.startswith('data/th/') and x.endswith('.parquet'))
    assert files_en and files_th, 'Berkas Parquet EN/TH tidak ditemukan. Cek struktur dataset sumber.'

    def cache_language(file_list, lang):
        paths = []
        for filename in file_list:
            print('Mengunduh/menggunakan cache', lang, filename)
            path = hf_hub_download(repo_id=REPO_ID, filename=filename,
                                   repo_type='dataset', revision=REV, local_dir=str(RAW))
            paths.append(str(Path(path)))
        return paths

    EN_FILES = cache_language(files_en, 'EN')
    TH_FILES = cache_language(files_th, 'TH')
    for f in EN_FILES + TH_FILES:
        print(f, round(Path(f).stat().st_size / 1024**2, 1), 'MB')

Mengunduh/menggunakan cache EN data/en/train.parquet


Mengunduh/menggunakan cache TH data/th/train.parquet


/content/wikimedia_wikidata_20260330_parquet/data/en/train.parquet 794.7 MB
/content/wikimedia_wikidata_20260330_parquet/data/th/train.parquet 12.1 MB


In [ ]:
if wt_valid():
    saved = json.loads(META.read_text(encoding='utf-8'))
    print('Dataset rekonstruksi valid sudah ada; tidak dibuat ulang.')
    print('Jumlah pasangan:', saved.get('pairs'))
else:
    assert 'EN_FILES' in globals(), 'Unduh parquet terlebih dahulu.'
    import duckdb
    con = duckdb.connect(database=':memory:')
    con.execute("SET memory_limit = '4GB'")
    tmpdb = Path('/content/wikititles_duckdb_tmp')
    tmpdb.mkdir(exist_ok=True)
    con.execute("SET temp_directory = '/content/wikititles_duckdb_tmp'")
    sql = """
    SELECT e.qid, e.title AS en_title, t.th_title
    FROM read_parquet(?) AS e
    INNER JOIN (
        SELECT qid, min(title) AS th_title
        FROM read_parquet(?)
        WHERE lang = 'th' AND qid IS NOT NULL AND title IS NOT NULL
        GROUP BY qid
    ) AS t ON e.qid=t.qid
    WHERE e.lang='en' AND e.title IS NOT NULL
    ORDER BY e.qid, e.title
    """
    result = con.execute(sql, [EN_FILES, TH_FILES])
    temp = OUT.with_suffix('.txt.part')
    seen = set()
    n = 0
    exclude_en = ('Category:', 'Template:', 'File:', 'Wikipedia:', 'Help:',
                  'Portal:', 'Draft:', 'Module:', 'User:', 'Talk:')
    exclude_th = ('หมวดหมู่:', 'ไฟล์:', 'แม่แบบ:', 'วิกิพีเดีย:')
    with temp.open('w', encoding='utf-8') as f:
        while True:
            rows = result.fetchmany(10000)
            if not rows: break
            for qid, en, th in rows:
                if not isinstance(qid, str) or not re.fullmatch(r'Q[0-9]+', qid): continue
                if qid in seen: continue
                if not isinstance(en, str) or not isinstance(th, str): continue
                en = ' '.join(en.split())
                th = ' '.join(th.split())
                if not en or not th or '||' in en or '||' in th: continue
                if en.startswith(exclude_en) or th.startswith(exclude_th): continue
                f.write(f'WTEN_TH_{qid}||{en}||{th}\n')
                seen.add(qid)
                n += 1
                if n % 25000 == 0: print('Pasangan valid:', format(n, ','))
    con.close()
    assert n > 0, 'Tidak ada pasangan: periksa struktur Parquet.'
    os.replace(temp, OUT)
    saved = {
        'source': REPO_ID, 'revision': REV,
        'language_pair': 'en-th', 'join_key': 'qid',
        'method': 'Wikimedia titles joined by same Wikidata QID; one pair per QID',
        'original_liu_dataset': False, 'paper_reported_pairs': 107398,
        'pairs': n, 'sha256': sha256(OUT),
        'format': 'WTEN_TH_QID||English title||Thai title',
        'note': 'Dataset rekonstruksi dari 2026, BUKAN 107398 pasangan asli Liu 2021.'
    }
    META.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding='utf-8')
    print('REKONSTRUKSI SELESAI:', format(n, ','), 'pasangan')
print('Output:', OUT)

Pasangan valid: 25,000
Pasangan valid: 50,000
Pasangan valid: 75,000
Pasangan valid: 100,000
Pasangan valid: 125,000
REKONSTRUKSI SELESAI: 147,100 pasangan
Output: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/source/wt_en_th_reconstructed_20260330.txt


In [ ]:
assert OUT.exists() and META.exists(), 'Dataset atau metadata belum tersimpan.'
meta=json.loads(META.read_text(encoding='utf-8'))
assert sha256(OUT)==meta['sha256'], 'Checksum tidak sesuai.'
with OUT.open(encoding='utf-8') as f:
    for i in range(5):
        row=f.readline().rstrip('\n')
        assert len(row.split('||'))==3, 'Format pasangan tidak valid'
        print(row)
print('Jumlah:',format(meta['pairs'],','),'(paper=107.398; belum tentu cocok)')
print('Metadata:',META)
print('Data ini REKONSTRUKSI 2026, bukan data asli Liu.')

WTEN_TH_Q1||Universe||เอกภพ
WTEN_TH_Q100||Boston||บอสตัน
WTEN_TH_Q1000||Gabon||ประเทศกาบอง
WTEN_TH_Q10000||Dutch Wikipedia||วิกิพีเดียภาษาดัตช์
WTEN_TH_Q1000022||Ningbo Rail Transit||รถไฟใต้ดินหนิงปัว
Jumlah: 147,100 (paper=107.398; belum tentu cocok)
Metadata: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/source/wt_en_th_reconstructed_20260330.metadata.json
Data ini REKONSTRUKSI 2026, bukan data asli Liu.


**Konversi ke format Liu.** Kode Liu (mode `--pairwise`) membaca setiap baris sebagai `label||nama1||nama2`, dan baris dengan label yang sama dianggap satu kelas. Mengikuti bagian 2.2 paper, setiap pasangan terjemahan diberi label buatan yang unik, sehingga setiap kelas hanya berisi dua kata. Data gabungan adalah MUSE disambung WikiTitles.

In [ ]:
import requests
SOURCE = DATA/'source'
SOURCE.mkdir(parents=True,exist_ok=True)
RAW_MUSE = MY_MUSE_PAIRS if MY_MUSE_PAIRS.exists() else SOURCE/'muse_en_th_original.txt'
print('Sumber MUSE:', RAW_MUSE)
RAW_WT = SOURCE / 'wt_en_th_reconstructed_20260330.txt'
MUSE = DATA/'muse_en_th_pairs.txt'
WT = DATA/'wt_en_th_pairs.txt'
BOTH = DATA/'wt_plus_muse_en_th_pairs.txt'


def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def pair_from_line(line, allow_space=False):
    line=line.strip()
    if not line: return None
    if '||' in line:
        p=line.split('||')
        if len(p)==3: return p[1].strip(),p[2].strip()
        if len(p)==2: return p[0].strip(),p[1].strip()
    if '\t' in line:
        p=line.split('\t',1);return p[0].strip(),p[1].strip()
    if allow_space:
        p=line.split()
        if len(p)==2: return p[0],p[1]
    return None

def convert_pairs(source, target, prefix, allow_space=False):
    meta=target.with_suffix('.metadata.json')
    source_hash=sha256(source)
    if target.exists() and meta.exists():
        m=json.loads(meta.read_text())
        if m.get('source_sha256')==source_hash and m.get('format_version')==2:
            return m['pairs']
    tmp=target.with_suffix('.part')
    n=0; skipped=0
    with Path(source).open('r',encoding='utf-8-sig') as inp, tmp.open('w',encoding='utf-8') as out:
        for line in inp:
            pair=pair_from_line(line,allow_space)
            if not pair or not all(pair) or any('||' in x or '\n' in x for x in pair):
                skipped+=1; continue
            en,th=pair
            n+=1
            out.write(f'{prefix}{n:09d}||{en}||{th}\n')
    assert n>0, 'Tidak ada pasangan valid pada '+str(source)
    os.replace(tmp,target)
    meta.write_text(json.dumps({'format_version':2,'source':str(source),
        'source_sha256':source_hash,'pairs':n,'skipped':skipped},indent=2))
    return n

if not RAW_MUSE.exists():
    url='https://dl.fbaipublicfiles.com/arrival/dictionaries/en-th.txt'
    resp=requests.get(url,timeout=180)
    resp.raise_for_status()
    assert len(resp.content)>1000,'Unduhan MUSE tidak valid'
    tmp=RAW_MUSE.with_suffix('.part'); tmp.write_bytes(resp.content); os.replace(tmp,RAW_MUSE)

count_muse=convert_pairs(RAW_MUSE,MUSE,'MUSE_TH_',allow_space=True)
print(f'MUSE: {count_muse:,} pasangan; paper=25.332', '(berbeda)' if count_muse!=25332 else '(jumlah cocok, isi belum terverifikasi)')

if RAW_WT.exists():
    count_wt=convert_pairs(RAW_WT,WT,'WT_TH_',allow_space=False)
    print(f'WikiTitles: {count_wt:,} pasangan; paper=107.398')
else:
    count_wt=0
    print('WikiTitles asli BELUM TERSEDIA. Varian wt dan gabungan tidak akan ditraining.')

if WT.exists() and count_wt:
    comb_meta=BOTH.with_suffix('.metadata.json')
    source_hash=sha256(MUSE)+':'+sha256(WT)
    if not (BOTH.exists() and comb_meta.exists() and json.loads(comb_meta.read_text()).get('source_sha256')==source_hash):
        temp=BOTH.with_suffix('.part')
        with temp.open('wb') as out:
            for p in (MUSE,WT):
                with p.open('rb') as inp: shutil.copyfileobj(inp,out)
        os.replace(temp,BOTH)
        comb_meta.write_text(json.dumps({'source_sha256':source_hash,'pairs':count_muse+count_wt},indent=2))
    print('Gabungan MUSE+WT:',count_muse+count_wt,'pasangan')

Sumber MUSE: /content/drive/MyDrive/sapbert_liu/data/muse/en_th_muse_pairs.txt
MUSE: 24,658 pasangan; paper=25.332 (berbeda)
WikiTitles: 147,100 pasangan; paper=107.398
Gabungan MUSE+WT: 171758 pasangan


In [ ]:
import torch

print("=== STATUS EXPERIMENT 2 ===")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK AKTIF")
print("Model Eksperimen 1:", "SIAP" if STAGE1_SRC is not None else "BELUM")
for name, path in [("MUSE", MUSE), ("WikiTitles", WT), ("Gabungan", BOTH)]:
    print(name, "SIAP" if path.exists() else "BELUM")
print("W&B:", "AKTIF" if WB_KEY else "NONAKTIF")
print("Batch:", BATCH_SIZE, "| loss:", LOSS_PARAMS, "| epoch:", EPOCHS, "| lr:", LR)

=== STATUS EXPERIMENT 2 ===
GPU: Tesla T4
Model Eksperimen 1: SIAP
MUSE SIAP
WikiTitles SIAP
Gabungan SIAP
W&B: AKTIF
Batch: 256 | loss: kode | epoch: 5 | lr: 2e-05


In [ ]:
datasets = {
    "MUSE": MUSE,
    "WikiTitles": WT,
    "Gabungan": BOTH
}

for name, path in datasets.items():
    if not path.exists():
        print(f"\n{name}: belum ada, dilewati")
        continue

    ids = set()
    pairs = set()
    total = 0

    with path.open(encoding="utf-8") as f:
        for line in f:
            fields = line.rstrip("\n").split("||")
            assert len(fields) == 3
            label, en, th = fields
            assert label and en and th

            ids.add(label)
            pairs.add((en, th))
            total += 1

    assert len(ids) == total, f"{name}: ada label ganda"
    print(f"\n{name}")
    print("Jumlah:", total)
    print("Label unik:", len(ids))
    print("Pasangan teks unik:", len(pairs))
    print("Duplikat pasangan:", total - len(pairs))
    print("Lokasi:", path)

print("\nVALIDASI STRUKTUR SELESAI")


MUSE
Jumlah: 24658
Label unik: 24658
Pasangan teks unik: 24658
Duplikat pasangan: 0
Lokasi: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/muse_en_th_pairs.txt

WikiTitles
Jumlah: 147100
Label unik: 147100
Pasangan teks unik: 147100
Duplikat pasangan: 0
Lokasi: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/wt_en_th_pairs.txt

Gabungan
Jumlah: 171758
Label unik: 171758
Pasangan teks unik: 171758
Duplikat pasangan: 0
Lokasi: /content/drive/MyDrive/sapbert_liu/experiment2_th/data/wt_plus_muse_en_th_pairs.txt

VALIDASI STRUKTUR SELESAI


In [ ]:
with MUSE.open(encoding='utf-8') as f:
    muse_pairs = [line.rstrip('\n').split('||')[1:] for line in f]
identik = sum(en == th for en, th in muse_pairs)
print(f'Total pasangan : {len(muse_pairs):,}')
print(f'Identik        : {identik:,} ({identik / len(muse_pairs):.1%})')
print(f'Terjemahan     : {len(muse_pairs) - identik:,}')

Dari 24.658 pasangan MUSE, sekitar 62% (15.317 pasangan) identik di kedua sisi, sehingga hanya sekitar 9.341 pasangan yang benar-benar menerjemahkan kata Inggris ke Thai. Kalau pasangan identik dibuang, data MUSE tersisa sekitar 9 ribu pasangan, jauh dari 25.332 yang dilaporkan paper. Karena itu kami menyimpulkan bahwa penulis kemungkinan juga memakai pasangan identik, dan kami mempertahankannya.

<a id="bab-7"></a>
## 7. Baseline

Ada dua baseline dalam reproduksi ini:

1. **Checkpoint resmi Liu dkk.** (`cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR`), dievaluasi dengan pipeline kami. Tujuannya memastikan pipeline evaluasi benar sebelum membandingkan model hasil training sendiri.
2. **Model Tahap 1 kami**, sebagai titik awal dan pembanding Tahap 2. Evaluasinya ada di Bab 9.2.

Metrik keduanya P@1 dan P@5 pada XL-BEL v0.0.

In [ ]:
EVAL = REPO / 'evaluation'
DICT = EVAL / 'xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt'
TEST_DE = EVAL / 'xl_bel/xlbel_v0.0/de_1k_test_query.txt'
for path in (DICT, TEST_DE):
    if not path.is_file():
        raise FileNotFoundError(f'Dataset belum ada: {path}')
    print(path.relative_to(REPO), 'bytes:', path.stat().st_size)

WRAPPER = REPO / 'src/model_wrapper.py'
src = WRAPPER.read_text(encoding='utf-8')
compile(src, str(WRAPPER), 'exec')
pattern = r'(def embed_dense\([^\n]*?batch_size=)(\d+)'
match = re.search(pattern, src)
if not match:
    raise RuntimeError('Tanda tangan embed_dense berbeda. Periksa file manual; patch TIDAK diterapkan.')
current_batch = int(match.group(2))
EVAL_BATCH = min(64, current_batch)
if current_batch > EVAL_BATCH:
    src = re.sub(pattern, lambda m: m.group(1) + str(EVAL_BATCH), src, count=1)
    WRAPPER.write_text(src, encoding='utf-8')
print('Batch inferensi default:', current_batch, '→', EVAL_BATCH)

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ID = 'cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR'
STORED_MODEL = STORE / 'models' / 'official_xlmr_2020ab'

def model_complete(p):
    return (p / 'config.json').is_file() and (bool(list(p.glob('*.safetensors'))) or bool(list(p.glob('*.bin'))))

if not model_complete(STORED_MODEL):
    print('Download checkpoint resmi ke Drive (sekali saja) ...')
    snapshot_download(repo_id=MODEL_ID, local_dir=str(STORED_MODEL))
if not model_complete(STORED_MODEL):
    raise RuntimeError('Checkpoint tidak lengkap; periksa unduhan model di Drive.')

LOCAL_MODEL = Path('/content/sapbert_official_model')
if not model_complete(LOCAL_MODEL):
    shutil.copytree(STORED_MODEL, LOCAL_MODEL, dirs_exist_ok=True)
print('Model evaluasi:', LOCAL_MODEL)

Download checkpoint resmi ke Drive (sekali saja) ...


Model evaluasi: /content/sapbert_official_model


In [ ]:
official_scores = {}
for lang in ('th', 'zh', 'de'):
    out_dir = STORE / 'results' / f'official_{lang}'
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['python', 'eval.py',
           '--model_dir', str(LOCAL_MODEL),
           '--dictionary_path', str(DICT),
           '--data_dir', str(EVAL / f'xl_bel/xlbel_v0.0/{lang}_1k_test_query.txt'),
           '--output_dir', str(out_dir),
           '--use_cuda', '--max_length', '25', '--custom_query_loader', '--agg_mode', 'cls']
    proc = subprocess.run(cmd, cwd=EVAL, capture_output=True, text=True)
    log = proc.stdout + proc.stderr
    if proc.returncode != 0:
        raise RuntimeError(log[-2000:])
    p1 = float(re.findall(r'acc@1=([0-9.]+)', log)[-1]) * 100
    p5 = float(re.findall(r'acc@5=([0-9.]+)', log)[-1]) * 100
    official_scores[lang] = (p1, p5)
    print(f'{lang}: P@1 {p1:.2f} | P@5 {p5:.2f}')

**Hasil baseline checkpoint resmi.** Skor berikut diperoleh dari run terpisah dengan perintah evaluasi yang sama; sel di atas belum dijalankan ulang di notebook gabungan ini.

| Bahasa | P@1 kami | P@5 kami | P@1 paper | P@5 paper |
|---|---:|---:|---:|---:|
| TH | 20,7 | 27,4 | 20,6 | 27,5 |
| ZH | 18,8 | 23,5 | 18,8 | 23,0 |
| DE | 33,1 | 37,1 | 31,8 | 37,3 |

Untuk TH dan ZH, skor checkpoint resmi hampir identik dengan paper, sehingga pipeline evaluasi kami dapat dipercaya. Model resmi ini dilatih dengan UMLS 2020AB, sedangkan paper menyebut 2020AA, yang dapat menjelaskan selisih kecil seperti pada DE.

Sebelum training penuh, kami juga menjalankan uji awal dengan 100 ribu pasangan dan batch 8. Hasilnya P@1/P@5 12,40/16,80 untuk TH dan 6,30/9,50 untuk ZH. Uji ini berbeda dari training penuh dalam jumlah data dan ukuran batch sekaligus, sehingga selisihnya tidak dapat diatribusikan ke salah satu faktor.

<a id="bab-8"></a>
## 8. Metode / Pendekatan yang Digunakan

### 8.1 Tahap 1: SAP dari XLM-R base

- **Arsitektur:** XLM-R base (12 lapisan Transformer, dimensi 768). Representasi sebuah nama diambil dari token [CLS] lapisan terakhir.
- **Input dan output:** input berupa pasangan nama yang memiliki CUI sama, output berupa encoder yang menempatkan nama-nama ber-CUI sama berdekatan di ruang vektor.
- **Loss:** Multi-Similarity loss dengan parameter dari kode resmi (α = 1, β = 60, base = 0,5), ditambah `TripletMarginMiner` (margin 0,2, tipe `all`). Contoh negatif diambil dari pasangan lain di batch yang sama.
- **Optimizer:** AdamW, learning rate 2e-5 konstan.
- **Hyperparameter:** batch 256 pasangan (512 nama per langkah), 1 epoch, panjang maksimal 25 token, AMP, seed 33. Semuanya mengikuti `train/pretrain.sh`, tanpa `--parallel` karena hanya ada satu GPU.
- **Alasan:** metode dan pengaturan dibuat sama dengan paper supaya hasilnya dapat dibandingkan langsung.

Tabel 10 paper menulis α = 2, β = 50, ε = 1, berbeda dengan nilai yang tertulis langsung di `src/metric_learning.py`. Nilai di kode tidak dapat diubah lewat skrip resmi, sehingga kami mengikuti kode. Tabel 10 juga menulis batch 512. Dalam mode pairwise, satu pasangan berisi dua nama, sehingga 256 pasangan sama dengan 512 nama per langkah. Kemungkinan inilah yang dimaksud paper.

**Training per shard.** Satu epoch berisi sekitar 82 ribu langkah, terlalu panjang untuk satu sesi Colab. Data dibagi menjadi 10 shard, dan setiap shard dilatih satu kali secara berurutan dengan bobot dari shard sebelumnya. Karena learning rate konstan, hasilnya mendekati satu epoch utuh. Perbedaannya hanya pada state AdamW yang di-reset di awal setiap shard, dan loss shard berikutnya tidak menunjukkan lonjakan setelah reset.

In [ ]:

from pathlib import Path
import os
import re
import sys
import json
import shutil
import subprocess
import time
import uuid
import torch
import wandb

assert "V" in globals(), "Pilih varian di Bab 6.1 terlebih dahulu."
STORE = Path("/content/drive/MyDrive/sapbert_liu")
REPO = Path("/content/reproduce_liu")
TRAIN = REPO / "train/train.py"
MODEL = STORE / "models/xlmr_base"
SHARD_DIR = V["shards"]
MANIFEST = V["manifest"]
RUN = V["run"]
PROGRESS = RUN / "progress.json"
FINAL = RUN / "final"
DONE = FINAL / "training_completed.json"
LOCAL_OUT = Path(f"/content/sap_{VARIANT}_out")
LOCAL_LOGS = Path(f"/content/sap_{VARIANT}_logs")

BATCH_SIZE = 256
LEARNING_RATE = 2e-5
MAX_LENGTH = 25
SEED = 33
MIN_FREE_DRIVE_GB = 2.5
assert torch.cuda.is_available(), "Aktifkan GPU terlebih dahulu!"
assert TRAIN.is_file(), "Kode Liu belum tersedia (Bab 4)."
assert MANIFEST.is_file(), f"Manifest shard {VARIANT} belum ada (Bab 6.1)."
assert (MODEL / "config.json").is_file(), "Config XLM-R belum tersedia di Drive."
assert list(MODEL.glob("*.safetensors")) or list(MODEL.glob("*.bin")), "Bobot XLM-R belum tersedia."

manifest = json.loads(MANIFEST.read_text())
N_SHARDS = manifest["n_shards"]
for k in range(N_SHARDS):
    assert (SHARD_DIR / f"shard_{k:02d}.txt").is_file(), f"Shard {VARIANT} belum ada di runtime ini (Bab 6.1)."

code = TRAIN.read_text(encoding="utf-8")
assert "num_workers=16" in code, "Versi train.py berbeda; periksa num_workers manual."
temporary = Path("/content/train_full_single_gpu.py")
temporary.write_text(code.replace("num_workers=16", "num_workers=0"), encoding="utf-8")

def model_saved(path):
    return (path / "config.json").is_file() and bool(
        list(path.glob("*.safetensors")) or list(path.glob("*.bin"))
    )

def save_progress(progress):
    part = PROGRESS.with_name("progress.json.part")
    part.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    part.replace(PROGRESS)

def drive_free_gb():
    return shutil.disk_usage("/content/drive").free / 1024**3

def last_tqdm_line(path):
    try:
        with path.open("rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 4000))
            tail = f.read().decode("utf-8", errors="replace")
    except FileNotFoundError:
        return ""
    lines = [s.strip() for s in re.split(r"[\r\n]", tail) if "it/s" in s or "s/it" in s]
    return lines[-1] if lines else "memuat data/model ..."

if DONE.is_file() and model_saved(FINAL):
    print(f"TRAINING {VARIANT} SUDAH SELESAI. Training tidak diulang.")
    print("Checkpoint:", FINAL)
else:
    RUN.mkdir(parents=True, exist_ok=True)
    if PROGRESS.is_file():
        progress = json.loads(PROGRESS.read_text())
        assert progress["n_shards"] == N_SHARDS, "Jumlah shard berubah sejak training dimulai."
    else:
        progress = {"variant": VARIANT, "n_shards": N_SHARDS, "completed_shards": 0, "checkpoint": None,
                    "shard_minutes": [],
                    "wandb_run_id": f"sap-{VARIANT.replace('_', '')}-" + uuid.uuid4().hex[:8]}
        save_progress(progress)

    for leftover in RUN.glob("ckpt_after_shard_*"):
        if leftover.name != progress["checkpoint"]:
            shutil.rmtree(leftover)
    if progress["checkpoint"]:
        assert model_saved(RUN / progress["checkpoint"]), "Checkpoint terakhir tidak lengkap; periksa manual."

    start = progress["completed_shards"]
    print(f"{VARIANT}: mulai dari shard {start + 1}/{N_SHARDS}" + (" (lanjutan)" if start else " (dari XLM-R Base)"))
    use_wandb = globals().get("LOG_TO_WANDB", True)
    if use_wandb:
        wandb.login()
    else:
        print("W&B dilewati: kurva Loss tidak dicatat, training tetap jalan.")
    LOCAL_LOGS.mkdir(exist_ok=True)
    (RUN / "logs").mkdir(exist_ok=True)

    for k in range(start, N_SHARDS):
        if drive_free_gb() < MIN_FREE_DRIVE_GB:
            raise RuntimeError(
                f"Ruang Drive tinggal {drive_free_gb():.1f} GB. Kosongkan Sampah Google Drive "
                "(checkpoint lama masuk ke sana), lalu jalankan ulang sel ini.")
        init_model = MODEL if progress["checkpoint"] is None else RUN / progress["checkpoint"]
        out = LOCAL_OUT / f"shard_{k:02d}"
        if out.exists():
            shutil.rmtree(out)
        log_file = LOCAL_LOGS / f"shard_{k:02d}.log"
        command = [
            sys.executable, str(temporary),
            "--model_dir", str(init_model),
            "--train_dir", str(SHARD_DIR / f"shard_{k:02d}.txt"),
            "--output_dir", str(out),
            "--use_cuda", "--amp", "--pairwise", "--use_miner",
            "--loss", "ms_loss", "--train_batch_size", str(BATCH_SIZE),
            "--epoch", "1", "--learning_rate", str(LEARNING_RATE),
            "--max_length", str(MAX_LENGTH), "--miner_margin", "0.2",
            "--type_of_triplets", "all", "--agg_mode", "cls",
            "--random_seed", str(SEED), "--checkpoint_step", "999999999"
        ]
        env = os.environ.copy()
        env.pop("WANDB_SERVICE", None)
        env["WANDB_MODE"] = "online" if use_wandb else "disabled"
        env["WANDB_NAME"] = f"sapbert_{VARIANT}_shard{k + 1:02d}"
        env["WANDB_RUN_GROUP"] = progress["wandb_run_id"]
        env["TOKENIZERS_PARALLELISM"] = "false"

        print(f"\n=== {VARIANT} shard {k + 1}/{N_SHARDS}: {manifest['counts'][k]:,} pasangan, "
              f"mulai dari {init_model.name} ===", flush=True)
        t0 = time.monotonic()
        with log_file.open("w", encoding="utf-8") as fh:
            process = subprocess.Popen(command, cwd=str(REPO / "train"), env=env,
                                       stdout=fh, stderr=subprocess.STDOUT)
            while True:
                try:
                    return_code = process.wait(timeout=300)
                    break
                except subprocess.TimeoutExpired:
                    minutes = (time.monotonic() - t0) / 60
                    print(f"  {minutes:.0f} menit | {last_tqdm_line(log_file)}", flush=True)
        shutil.copy2(log_file, RUN / "logs" / log_file.name)
        if return_code != 0:
            print(log_file.read_text(errors="replace")[-4000:])
            raise RuntimeError(f"{VARIANT} shard {k + 1} gagal. Progress tetap {k}/{N_SHARDS}; "
                               "perbaiki error lalu jalankan ulang sel ini.")
        if not model_saved(out):
            raise RuntimeError(f"Shard {k + 1} selesai tetapi bobot model tidak ditemukan di {out}.")

        name = f"ckpt_after_shard_{k + 1:02d}"
        part = RUN / (name + ".part")
        if part.exists():
            shutil.rmtree(part)
        shutil.copytree(out, part)
        part.rename(RUN / name)
        previous = progress["checkpoint"]
        progress["completed_shards"] = k + 1
        progress["checkpoint"] = name
        progress["shard_minutes"].append(round((time.monotonic() - t0) / 60, 1))
        save_progress(progress)
        if previous:
            shutil.rmtree(RUN / previous)
        shutil.rmtree(out)

        avg = sum(progress["shard_minutes"]) / len(progress["shard_minutes"])
        print(f"{VARIANT} shard {k + 1}/{N_SHARDS} selesai ({progress['shard_minutes'][-1]} menit). "
              f"Perkiraan sisa: {avg * (N_SHARDS - k - 1) / 60:.1f} jam.", flush=True)

    last = RUN / progress["checkpoint"]
    try:
        sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True,
                                      stderr=subprocess.DEVNULL).strip()
    except Exception:
        sha = "unknown"
    (last / "training_completed.json").write_text(json.dumps({
        "status": "completed", "variant": VARIANT, "pairs": manifest["total_pairs"],
        "n_shards": N_SHARDS, "shard_seed": manifest["seed"], "epochs": 1, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "max_length": MAX_LENGTH, "miner_margin": 0.2,
        "loss_impl": "ms_loss as released (alpha=1, beta=60, base=0.5)",
        "model_base": "xlm-roberta-base",
        "data": V["data_note"],
        "optimizer_state_reset_per_shard": True,
        "shard_minutes": progress["shard_minutes"], "wandb_run_id": progress["wandb_run_id"],
        "git_commit": sha,
        "note": "SAP penuh; satu GPU tanpa --parallel; AdamW di-reset di awal tiap shard",
    }, indent=2), encoding="utf-8")
    assert not FINAL.exists(), f"{FINAL} sudah ada tanpa penanda selesai; periksa manual."
    last.rename(FINAL)
    progress["checkpoint"] = FINAL.name
    save_progress(progress)
    print(f"\nTRAINING SAP {VARIANT} SELESAI!")
    print("Checkpoint:", FINAL)
    print("Total waktu (jam):", round(sum(progress["shard_minutes"]) / 60, 1))

all_syn: mulai dari shard 6/10 (lanjutan)

=== all_syn shard 6/10: 2,108,028 pasangan, mulai dari ckpt_after_shard_05 ===
  5 menit | 7%|▋         | 563/8235 [03:47<51:51,  2.47it/s]
  10 menit | 16%|█▌        | 1305/8235 [08:47<46:43,  2.47it/s]
  15 menit | 25%|██▍       | 2048/8235 [13:47<41:50,  2.46it/s]
  20 menit | 34%|███▍      | 2789/8235 [18:47<36:52,  2.46it/s]
  25 menit | 43%|████▎     | 3531/8235 [23:47<31:40,  2.47it/s]
  30 menit | 52%|█████▏    | 4272/8235 [28:47<26:45,  2.47it/s]
  35 menit | 61%|██████    | 5013/8235 [33:47<21:45,  2.47it/s]
  40 menit | 70%|██████▉   | 5755/8235 [38:47<16:40,  2.48it/s]
  45 menit | 79%|███████▉  | 6496/8235 [43:47<11:46,  2.46it/s]
  50 menit | 88%|████████▊ | 7237/8235 [48:47<06:44,  2.47it/s]
  55 menit | 97%|█████████▋| 7978/8235 [53:47<01:43,  2.48it/s]
all_syn shard 6/10 selesai (57.0 menit). Perkiraan sisa: 3.8 jam.

=== all_syn shard 7/10: 2,108,470 pasangan, mulai dari ckpt_after_shard_06 ===
  5 menit | 9%|▊         | 711/

### 8.2 Tahap 2: pelatihan lanjut dengan data paralel

- **Model awal:** checkpoint final Tahap 1.
- **Input:** pasangan terjemahan Inggris–Thai dengan label unik per pasangan, sehingga kata Inggris dan terjemahannya ditarik berdekatan, sedangkan kata dari pasangan lain menjadi contoh negatif.
- **Loss, optimizer, dan miner:** sama dengan Tahap 1.
- **Hyperparameter:** 5 epoch, batch 256, learning rate 2e-5, panjang maksimal 25, margin miner 0,2, seed 33, sesuai `train/xling_train.sh` dan Tabel 10.
- **Varian:** + MUSE, + WikiTitles, dan + gabungan keduanya. Setiap varian dilatih terpisah dari model Tahap 1 yang sama.

Sel berikut memeriksa model Tahap 1, menyiapkan fungsi evaluasi dan sidik jari bobot (dipakai lagi di Bab 9), lalu mendefinisikan fungsi training.

In [ ]:
def is_model(path):
    p = Path(path)
    return (p / 'config.json').is_file() and (any(p.glob('*.safetensors')) or any(p.glob('*.bin')))

STAGE1_SRC = MY_STAGE1 if is_model(MY_STAGE1) else None
STAGE1_LABEL = 'ngurah_XLMR_SAP_all_syn_2020AA'

if STAGE1_SRC is None:
    print('MODEL EKSPERIMEN 1 BELUM TERBACA di', MY_STAGE1)
    print('Pastikan Drive yang di-mount adalah akun tempat training penuh dijalankan.')
else:
    config = json.loads((STAGE1_SRC / 'config.json').read_text())
    meta_file = STAGE1_SRC / 'training_completed.json'
    metadata = json.loads(meta_file.read_text()) if meta_file.exists() else {}
    assert config.get('model_type') == 'xlm-roberta', 'Checkpoint bukan XLM-R!'
    assert config.get('hidden_size') == 768, 'Bukan XLM-R Base sesuai target.'
    assert metadata.get('variant') == 'all_syn', 'Bukan checkpoint SAP all syn!'
    assert metadata.get('status') == 'completed', 'Penanda training belum selesai!'
    print('Checkpoint siap:', STAGE1_SRC)
    print('Model:', config['model_type'], '| hidden:', config['hidden_size'])
    print('Training:', {k: metadata.get(k) for k in ['variant', 'pairs', 'batch_size', 'epochs', 'loss_impl']})

def ensure_stage1_local():
    assert STAGE1_SRC is not None, 'Model Tahap 1 belum tersedia.'
    signature_file = LOCAL_STAGE1 / '.source_info.json'
    weights = sorted(list(STAGE1_SRC.glob('*.safetensors')) + list(STAGE1_SRC.glob('*.bin')))
    signature = {'source': str(STAGE1_SRC), 'weight_sizes': [p.stat().st_size for p in weights]}
    if not (is_model(LOCAL_STAGE1) and signature_file.exists() and
            json.loads(signature_file.read_text()) == signature):
        if LOCAL_STAGE1.exists(): shutil.rmtree(LOCAL_STAGE1)
        shutil.copytree(STAGE1_SRC, LOCAL_STAGE1)
        signature_file.write_text(json.dumps(signature))
    return LOCAL_STAGE1

Checkpoint siap: /content/drive/MyDrive/sapbert_liu/runs/sapbert_full_umls2020aa/final
Model: xlm-roberta | hidden: 768
Training: {'variant': 'all_syn', 'pairs': 21082913, 'batch_size': 256, 'epochs': 1, 'loss_impl': 'ms_loss as released (alpha=1, beta=60, base=0.5)'}


In [ ]:
from datetime import datetime, timezone
XLBEL_EVAL = REPO/'evaluation'
XLBEL_DICT = XLBEL_EVAL/'xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt'
XLBEL_TEST_TH = XLBEL_EVAL/'xl_bel/xlbel_v0.0/th_1k_test_query.txt'
assert XLBEL_DICT.is_file() and XLBEL_TEST_TH.is_file(), 'Benchmark XL-BEL TH belum ada di repository'

def gpu_info():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Aktifkan Runtime > Change runtime type > GPU (T4/L4/A100)')
    return torch.cuda.get_device_properties(0).total_memory/1024**3

def weight_fingerprint(p):
    weights=sorted(list(Path(p).glob('*.safetensors'))+list(Path(p).glob('*.bin')))
    assert weights, 'Bobot tidak ada pada '+str(p)
    return {'files':[[x.name,x.stat().st_size] for x in weights],
            'sha256':[sha256(x) for x in weights]}

def wb_eval(name, payload):
    if not WB_KEY:
        print('W&B dilewati (WANDB_API_KEY tidak tersedia).')
        return
    import wandb
    run=wandb.init(project=WANDB_PROJECT,group=WANDB_GROUP,job_type='evaluation',
                   name='XL-BEL_TH_'+name,id=hashlib.md5(('xlbelth:'+WANDB_GROUP+name).encode()).hexdigest()[:16],resume='allow',reinit=True,
                   config={'variant':name,'language':'th','checkpoint':payload['model_source'],
                           'git_commit':COMMIT,'experimental_scope':'experimental reproduction'})
    run.log({'eval/P@1_percent':payload['P@1'],'eval/P@5_percent':payload['P@5'],
             'eval/acc@1':payload['P@1']/100,'eval/acc@5':payload['P@5']/100})
    run.summary['P@1_percent']=payload['P@1'];run.summary['P@5_percent']=payload['P@5']
    run.finish()

def eval_variant(name, model_path, source_label):
    model_path=Path(model_path)
    assert is_model(model_path), 'Checkpoint tidak valid: '+str(model_path)
    result_dir=RESULTS/name
    result_dir.mkdir(parents=True,exist_ok=True)
    file=result_dir/'result.json'
    fp=weight_fingerprint(model_path)
    if file.exists():
        d=json.loads(file.read_text())
        if d.get('model_fingerprint')==fp and d.get('git_commit')==COMMIT:
            print(f'{name}: memakai skor tersimpan {d["P@1"]:.2f}% / {d["P@5"]:.2f}%')
            wb_eval(name,d)
            return d
    gpu_info()
    cmd=[sys.executable,'eval.py','--model_dir',str(model_path),
         '--dictionary_path',str(XLBEL_DICT),'--data_dir',str(XLBEL_TEST_TH),
         '--output_dir',str(result_dir),'--use_cuda','--max_length','25',
         '--custom_query_loader','--agg_mode','cls']
    logfile=result_dir/'eval.log'
    print('Evaluasi XL-BEL TH:',name)
    with logfile.open('w') as out:
        p=subprocess.run(cmd,cwd=str(XLBEL_EVAL),stdout=out,stderr=subprocess.STDOUT)
    output=logfile.read_text(errors='replace')
    if p.returncode:
        print(output[-4500:]);raise RuntimeError('Evaluasi gagal. Lihat log di Drive.')
    a1=re.findall(r'acc@1\s*=\s*([0-9.]+)',output)
    a5=re.findall(r'acc@5\s*=\s*([0-9.]+)',output)
    if not a1 or not a5: raise RuntimeError('acc@1/acc@5 tidak ditemukan di log.')
    d={'variant':name,'language':'th','P@1':float(a1[-1])*100,'P@5':float(a5[-1])*100,
       'model_source':source_label,'model_fingerprint':fp,'git_commit':COMMIT,
       'evaluation_max_length':25,'agg_mode':'cls',
       'created_utc':datetime.now(timezone.utc).isoformat()}
    file.write_text(json.dumps(d,indent=2,ensure_ascii=False))
    print(f'{name}: P@1={d["P@1"]:.2f}%  P@5={d["P@5"]:.2f}%')
    wb_eval(name,d)
    return d

In [ ]:
def train_variant(name, dataset_path):
    dataset_path = Path(dataset_path)
    assert STAGE1_SRC is not None, 'Model Tahap 1 belum tersedia.'
    assert dataset_path.is_file(), f'Dataset tidak ditemukan: {dataset_path}'
    gpu_info()
    run = run_name(name)
    run_dir = RUNS / run
    final = run_dir / 'final'
    done = final / 'training_completed.json'
    with dataset_path.open(encoding='utf-8') as f:
        n_pairs = sum(1 for _ in f)
    cfg = json.loads(json.dumps({
        'trainer': 'official train.py (argumen train/xling_train.sh, tanpa --parallel)',
        'source_label': STAGE1_LABEL, 'source_fingerprint': weight_fingerprint(STAGE1_SRC),
        'data': dataset_path.name, 'data_sha256': sha256(dataset_path), 'pairs': n_pairs,
        'batch_size': BATCH_SIZE, 'lr': LR, 'epoch': EPOCHS, 'max_length': MAX_LEN, 'seed': SEED,
        'miner_margin': MINER_MARGIN, 'loss_params': LOSS_PARAMS,
        'ms_loss': {'alpha': 1, 'beta': 60, 'base': 0.5} if LOSS_PARAMS == 'kode'
                   else {'alpha': 2, 'beta': 50, 'base': 1},
        'git_commit': COMMIT}))

    if done.exists() and is_model(final):
        old = json.loads(done.read_text()).get('config', {})
        if old != cfg:
            diff = {k: {'tersimpan': old.get(k), 'sekarang': cfg.get(k)}
                    for k in set(old) | set(cfg) if old.get(k) != cfg.get(k)}
            raise RuntimeError(f'Model final {run} berasal dari konfigurasi lain: {diff}. '
                               f'Hapus folder {final} atau pakai nama varian lain.')
        print(run, ': FINAL sudah ada dengan konfigurasi yang sama; training dilewati.')
        return final

    free_gb = shutil.disk_usage('/content/drive').free / 1024**3
    assert free_gb > 1.5, f'Ruang Drive tinggal {free_gb:.1f} GB; kosongkan Sampah Drive dulu.'
    source = ensure_stage1_local()

    code = (REPO / 'train/train.py').read_text(encoding='utf-8')
    assert 'num_workers=16' in code, 'Versi train.py berbeda; periksa num_workers manual.'
    assert code.count('wandb.init(project="sapbert")') == 1, 'Baris wandb.init di train.py berubah.'
    assert code.count('    if args.parallel:') == 1, 'Struktur main() di train.py berubah.'
    code = code.replace('num_workers=16', 'num_workers=0')
    code = code.replace('wandb.init(project="sapbert")',
                        'wandb.init(project=os.environ.get("WANDB_PROJECT", "sapbert"))')
    if LOSS_PARAMS == 'paper':
        code = code.replace('    if args.parallel:',
            '    from pytorch_metric_learning import losses as pml_losses\n'
            '    model.loss = pml_losses.MultiSimilarityLoss(alpha=2, beta=50, base=1)  # ablasi: parameter teks paper\n'
            '    LOGGER.info("MS loss diganti ke parameter paper: alpha=2 beta=50 base=1")\n'
            '    if args.parallel:')
    temporary = Path('/content/train_exp2_single_gpu.py')
    temporary.write_text(code, encoding='utf-8')

    out = LOCAL_OUT / run
    if out.exists(): shutil.rmtree(out)
    command = [sys.executable, str(temporary),
        '--model_dir', str(source), '--train_dir', str(dataset_path), '--output_dir', str(out),
        '--use_cuda', '--epoch', str(EPOCHS), '--train_batch_size', str(BATCH_SIZE),
        '--learning_rate', str(LR), '--max_length', str(MAX_LEN), '--checkpoint_step', '999999',
        '--amp', '--pairwise', '--random_seed', str(SEED), '--loss', 'ms_loss', '--use_miner',
        '--type_of_triplets', 'all', '--miner_margin', str(MINER_MARGIN), '--agg_mode', 'cls']
    if SAVE_EPOCH_CHECKPOINTS:
        command.append('--save_checkpoint_all')
    env = os.environ.copy()
    env.pop('WANDB_SERVICE', None)
    env['WANDB_PROJECT'] = WANDB_PROJECT
    env['WANDB_RUN_GROUP'] = WANDB_GROUP
    env['WANDB_NAME'] = 'exp2_TH_' + run
    env['WANDB_JOB_TYPE'] = 'bitext-training'
    env['TOKENIZERS_PARALLELISM'] = 'false'

    steps = math.ceil(n_pairs / BATCH_SIZE) * EPOCHS
    log_file = Path('/content') / f'train_{run}.log'
    print(f'{run}: {n_pairs:,} pasangan, {EPOCHS} epoch, batch {BATCH_SIZE}, ±{steps:,} langkah', flush=True)
    t0 = time.monotonic()
    with log_file.open('w', encoding='utf-8') as fh:
        proc = subprocess.Popen(command, cwd=str(REPO / 'train'), env=env, stdout=fh, stderr=subprocess.STDOUT)
        while True:
            try:
                rc = proc.wait(timeout=120)
                break
            except subprocess.TimeoutExpired:
                tail = log_file.read_bytes()[-3000:].decode('utf-8', 'replace')
                prog = [s.strip() for s in re.split(r'[\r\n]', tail) if 'it/s' in s or 's/it' in s]
                print(f'  {(time.monotonic() - t0) / 60:.0f} menit | {prog[-1] if prog else "memuat data/model ..."}',
                      flush=True)
    if rc != 0:
        print(log_file.read_text(errors='replace')[-4000:])
        raise RuntimeError(f'Training {run} gagal; lihat akhir log di atas.')
    assert is_model(out), f'Training selesai tetapi bobot final tidak ada di {out}'
    if SAVE_EPOCH_CHECKPOINTS:
        missing = [e for e in range(1, EPOCHS + 1) if not is_model(out / f'checkpoint_{e}')]
        if missing: print('Peringatan: checkpoint per epoch tidak lengkap:', missing)
    minutes = round((time.monotonic() - t0) / 60, 1)

    run_dir.mkdir(parents=True, exist_ok=True)
    part = run_dir / 'final.part'
    if part.exists(): shutil.rmtree(part)
    shutil.copytree(out, part, ignore=shutil.ignore_patterns('checkpoint_*'))
    shutil.copy2(log_file, part / 'training.log')
    (part / 'training_completed.json').write_text(json.dumps({
        'complete': True, 'config': cfg, 'minutes': minutes,
        'epoch_checkpoints_local': str(out) if SAVE_EPOCH_CHECKPOINTS else None,
        'note': 'Tahap 2 Liu (Tabel 3) dengan train.py resmi; satu GPU tanpa --parallel'},
        indent=2, ensure_ascii=False), encoding='utf-8')
    if final.exists(): shutil.rmtree(final)
    os.replace(part, final)
    print(f'FINAL SELESAI ({minutes} menit):', final)
    return final

In [ ]:
if STAGE1_SRC is None:
    print('Model Tahap 1 belum tersedia.')
else:
    train_variant('muse', MUSE)

muse: 24,658 pasangan, 5 epoch, batch 256, ±485 langkah
  2 menit | 85%|████████▍ | 82/97 [00:48<00:09,  1.66it/s]
  4 menit | 35%|███▌      | 34/97 [00:20<00:38,  1.62it/s]
  6 menit | 98%|█████████▊| 95/97 [00:59<00:01,  1.61it/s]
FINAL SELESAI (6.6 menit): /content/drive/MyDrive/sapbert_liu/experiment2_th/runs/muse/final


In [ ]:
if STAGE1_SRC is None:
    print('Model Tahap 1 belum tersedia.')
elif not WT.exists():
    print('Data WikiTitles belum tersedia.')
else:
    train_variant('wikititles', WT)

wikititles: 147,100 pasangan, 5 epoch, batch 256, ±2,875 langkah
  2 menit | 30%|███       | 173/575 [01:49<04:12,  1.59it/s]
  4 menit | 63%|██████▎   | 361/575 [03:49<02:17,  1.56it/s]
  6 menit | 95%|█████████▌| 549/575 [05:49<00:16,  1.55it/s]
  8 menit | 26%|██▋       | 151/575 [01:36<04:32,  1.56it/s]
  10 menit | 59%|█████▉    | 339/575 [03:36<02:30,  1.57it/s]
  12 menit | 92%|█████████▏| 528/575 [05:36<00:29,  1.57it/s]
  14 menit | 22%|██▏       | 128/575 [01:21<04:40,  1.59it/s]
  16 menit | 55%|█████▍    | 316/575 [03:21<02:45,  1.57it/s]
  18 menit | 88%|████████▊ | 505/575 [05:21<00:44,  1.57it/s]
  20 menit | 14%|█▍        | 83/575 [00:53<05:17,  1.55it/s]
  22 menit | 47%|████▋     | 273/575 [02:53<03:11,  1.58it/s]
  24 menit | 80%|████████  | 461/575 [04:53<01:12,  1.57it/s]
  26 menit | 11%|█         | 64/575 [00:41<05:30,  1.55it/s]
  28 menit | 44%|████▍     | 253/575 [02:41<03:23,  1.58it/s]
  30 menit | 77%|███████▋  | 442/575 [04:41<01:24,  1.57it/s]
  32 menit 

In [ ]:
if STAGE1_SRC is None:
    print('Model Tahap 1 belum tersedia.')
elif not BOTH.exists():
    print('Data gabungan belum tersedia.')
else:
    train_variant('wt_plus_muse', BOTH)

wt_plus_muse: 171,758 pasangan, 5 epoch, batch 256, ±3,355 langkah
  2 menit | 26%|██▌       | 176/671 [01:48<05:26,  1.52it/s]
  4 menit | 54%|█████▍    | 365/671 [03:48<03:13,  1.58it/s]
  6 menit | 83%|████████▎ | 554/671 [05:48<01:14,  1.58it/s]
  8 menit | 6%|▌         | 41/671 [00:25<06:48,  1.54it/s]
  10 menit | 34%|███▍      | 230/671 [02:26<04:38,  1.58it/s]
  12 menit | 62%|██████▏   | 419/671 [04:25<02:39,  1.58it/s]
  14 menit | 91%|█████████ | 609/671 [06:25<00:39,  1.59it/s]
  16 menit | 16%|█▌        | 109/671 [01:09<05:52,  1.59it/s]
  18 menit | 45%|████▍     | 299/671 [03:09<03:54,  1.59it/s]
  20 menit | 73%|███████▎  | 489/671 [05:09<01:56,  1.57it/s]
  22 menit | 100%|██████████| 671/671 [07:04<00:00,  1.58it/s]
  24 menit | 26%|██▋       | 177/671 [01:52<05:15,  1.56it/s]
  26 menit | 55%|█████▍    | 367/671 [03:52<03:12,  1.58it/s]
  28 menit | 83%|████████▎ | 557/671 [05:52<01:12,  1.58it/s]
  30 menit | 8%|▊         | 54/671 [00:34<06:44,  1.53it/s]
  32 menit

<a id="bab-9"></a>
## 9. Validasi dan Evaluasi

- **Data uji:** XL-BEL v0.0, 1.000 frasa per bahasa. Versi ini yang dipakai untuk angka di paper.
- **Kamus target:** `umls_onto_all_lang_cased_wikimed_only_399931.txt` (399.931 nama).
- **Prosedur:** `evaluation/eval.py` resmi. Frasa dan seluruh nama di kamus diubah menjadi vektor [CLS] (panjang maksimal 25 token), lalu 10 nama dengan skor *dot product* tertinggi diambil sebagai kandidat.
- **Metrik:** P@k, disebut `acc@k` di kode. Sebuah frasa dihitung benar jika setidaknya satu dari k kandidat teratas memiliki CUI yang benar. Frasa dengan beberapa CUI benar dihitung benar jika salah satunya cocok. Definisi ini berbeda dari *precision@k* pada *information retrieval*.
- **Pemilihan model:** tidak ada pemilihan berdasarkan data uji. Hyperparameter mengikuti paper, dan model yang dilaporkan adalah model di akhir training (1 epoch untuk Tahap 1, 5 epoch untuk Tahap 2).
- **Seed:** 33, satu kali run per konfigurasi.

**Interval kepercayaan.** Dengan n = 1.000 dan P@1 = p, galat baku adalah √(p(1 − p)/n). Untuk p = 0,21 nilainya sekitar 1,3 poin, sehingga interval kepercayaan 95% sekitar ±2,5 poin. Untuk p = 0,30 intervalnya sekitar ±2,8 poin. Selisih dengan paper di bawah batas ini dianggap masih dalam noise statistik.

### 9.1 Evaluasi Tahap 1 (TH dan ZH)

In [ ]:
from pathlib import Path
import subprocess
import re
import json
import sys
import torch

assert "V" in globals(), "Pilih varian di Bab 6.1 terlebih dahulu."
STORE = Path("/content/drive/MyDrive/sapbert_liu")
REPO = Path("/content/reproduce_liu")
MODEL = V["run"] / "final"

LANGS = ["th", "zh"]
EVAL = REPO / "evaluation"
DICT = EVAL / "xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt"

PAPER = {
    "en_syn": {"en": (78.1, 80.9), "es": (47.9, 53.5), "de": (27.6, 32.0), "fi": (12.2, 14.7), "ru": (21.8, 25.9),
               "tr": (29.3, 35.9), "ko": (4.5, 6.7), "zh": (7.9, 11.3), "ja": (8.3, 11.3), "th": (11.5, 16.2)},
    "all_syn": {"en": (78.2, 81.0), "es": (56.4, 62.7), "de": (31.8, 37.3), "fi": (18.6, 22.2), "ru": (35.4, 41.2),
                "tr": (42.8, 48.9), "ko": (16.7, 21.4), "zh": (18.8, 23.0), "ja": (24.0, 28.1), "th": (20.6, 27.5)},
}

assert (MODEL / "training_completed.json").exists(), f"Training {VARIANT} belum selesai."
assert DICT.exists(), "Kamus evaluasi belum ada"
assert (EVAL / "eval.py").exists(), "Repository belum siap"

all_scores = {}
for LANG in LANGS:
    TEST = EVAL / f"xl_bel/xlbel_v0.0/{LANG}_1k_test_query.txt"
    RESULT = STORE / f"results/{V['result_prefix']}_{LANG}"
    RESULT.mkdir(parents=True, exist_ok=True)
    result_file = RESULT / "result.json"
    assert TEST.exists(), f"Data uji {LANG} belum ada"

    if result_file.exists():
        scores = json.loads(result_file.read_text())
        print(f"{LANG}: hasil evaluasi sebelumnya ditemukan.")
    else:
        assert torch.cuda.is_available(), "Aktifkan GPU Colab"

        wrapper = REPO / "src/model_wrapper.py"
        code = wrapper.read_text()
        pattern = r"(def embed_dense\([^\n]*?batch_size=)(\d+)"
        m = re.search(pattern, code)
        assert m, "Periksa fungsi embed_dense"
        if int(m.group(2)) > 64:
            code = re.sub(pattern, lambda x: x.group(1)+"64", code, count=1)
            wrapper.write_text(code)

        command = [
            sys.executable, "eval.py",
            "--model_dir", str(MODEL),
            "--dictionary_path", str(DICT),
            "--data_dir", str(TEST),
            "--output_dir", str(RESULT),
            "--use_cuda",
            "--max_length", "25",
            "--custom_query_loader",
            "--agg_mode", "cls"
        ]

        print(f"Memulai evaluasi XL-BEL {LANG} untuk {VARIANT}")
        log_file = RESULT / "eval.log"

        with log_file.open("w") as log:
            process = subprocess.Popen(
                command,
                cwd=str(EVAL),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True
            )
            for line in process.stdout:
                log.write(line)
                if "acc@" in line or "embedding dictionary" in line:
                    print(line.strip())
            rc = process.wait()

        if rc != 0:
            raise RuntimeError(f"Evaluasi gagal. Periksa {log_file}")

        output = log_file.read_text(errors="replace")
        a1 = re.findall(r"acc@1=([0-9.]+)", output)
        a5 = re.findall(r"acc@5=([0-9.]+)", output)

        assert a1 and a5, "Skor tidak ditemukan di log"

        scores = {
            "model": f"sapbert_{VARIANT}_umls2020aa",
            "variant": VARIANT,
            "language": LANG,
            "P@1": float(a1[-1]) * 100,
            "P@5": float(a5[-1]) * 100
        }
        result_file.write_text(json.dumps(scores, indent=2))
    all_scores[LANG] = scores

print(f"\n=== HASIL EVALUASI {VARIANT} (kita vs paper) ===")
for LANG, scores in all_scores.items():
    p1, p5 = PAPER[VARIANT][LANG]
    print(f"{LANG}: P@1 {scores['P@1']:.2f}% (paper {p1}) | P@5 {scores['P@5']:.2f}% (paper {p5})")
print("Tersimpan di:", STORE / "results")

Memulai evaluasi XL-BEL th untuk all_syn
[start embedding dictionary...]
10/26/2026 05:59:20 AM: [ acc@1=0.211 ]
10/26/2026 05:59:20 AM: [ acc@5=0.269 ]
Memulai evaluasi XL-BEL zh untuk all_syn
[start embedding dictionary...]
10/26/2026 06:04:20 AM: [ acc@1=0.18 ]
10/26/2026 06:04:20 AM: [ acc@5=0.224 ]

=== HASIL EVALUASI all_syn (kita vs paper) ===
th: P@1 21.10% (paper 20.6) | P@5 26.90% (paper 27.5)
zh: P@1 18.00% (paper 18.8) | P@5 22.40% (paper 23.0)
Tersimpan di: /content/drive/MyDrive/sapbert_liu/results


### 9.2 Evaluasi Tahap 2 (TH)

Model Tahap 1 dievaluasi ulang di sesi Tahap 2 sebagai baseline. Skornya harus sama dengan Bab 9.1, yang sekaligus memastikan model yang dimuat benar.

In [ ]:
if STAGE1_SRC is None:
    print('Model Tahap 1 belum tersedia.')
else:
    stage1_local=ensure_stage1_local()
    baseline=eval_variant('baseline_SAP_all_syn_TH',stage1_local,STAGE1_LABEL)
    target=(20.6,27.5)
    print('Paper baseline: P@1=20.6%, P@5=27.5%')
    print('Selisih (poin):',round(baseline['P@1']-target[0],2),round(baseline['P@5']-target[1],2))

Evaluasi XL-BEL TH: baseline_SAP_all_syn_TH
baseline_SAP_all_syn_TH: P@1=21.10%  P@5=26.90%


Paper baseline: P@1=20.6%, P@5=27.5%
Selisih (poin): 0.5 -0.6


In [ ]:
for variant in ('muse', 'wikititles', 'wt_plus_muse'):
    run = run_name(variant)
    final = RUNS / run / 'final'
    if (final / 'training_completed.json').exists() and is_model(final):
        eval_variant(run, final, f'{STAGE1_LABEL}+{run}')
    else:
        print(run, ': belum punya final lengkap; evaluasi dilewati')

Evaluasi XL-BEL TH: muse
muse: P@1=25.50%  P@5=32.30%


Evaluasi XL-BEL TH: wikititles
wikititles: P@1=29.70%  P@5=37.40%


Evaluasi XL-BEL TH: wt_plus_muse
wt_plus_muse: P@1=30.40%  P@5=37.40%


<a id="bab-10"></a>
## 10. Hasil Eksperimen dan Analisis

**Tahap 1: XLMR + SAP_all_syn (Tabel 2 paper)**

| Model | TH P@1 | TH P@5 | ZH P@1 | ZH P@5 | Catatan |
|---|---:|---:|---:|---:|---|
| Paper | 20,6 | 27,5 | 18,8 | 23,0 | Tabel 2 |
| Checkpoint resmi (Bab 7) | 20,7 | 27,4 | 18,8 | 23,5 | validasi pipeline |
| **Training ulang kami** | **21,10** | **26,90** | **18,00** | **22,40** | selisih TH +0,5 / −0,6; ZH −0,8 / −0,6 |

**Tahap 2: data terjemahan EN–TH (Tabel 3 paper, Thai)**

| Kondisi | P@1 paper | P@5 paper | P@1 kami | P@5 kami | Selisih P@1 | Kenaikan P@1 dari baseline (kami / paper) |
|---|---:|---:|---:|---:|---:|---:|
| SAP all_syn (baseline) | 20,6 | 27,5 | 21,10 | 26,90 | +0,5 | – |
| + MUSE | 25,9 | 32,8 | 25,50 | 32,30 | −0,4 | +4,4 / +5,3 |
| + WikiTitles | 30,0 | 37,6 | 29,70 | 37,40 | −0,3 | +8,6 / +9,4 |
| + WikiTitles + MUSE | 30,9 | 37,9 | 30,40 | 37,40 | −0,5 | +9,3 / +10,3 |

Sel berikut menyusun tabel Tahap 2 dari berkas hasil di Drive dan menyimpannya sebagai `reports/table3_th.csv`.

In [ ]:
PAPER = {'baseline_SAP_all_syn_TH': (20.6, 27.5), 'muse': (25.9, 32.8),
         'wikititles': (30.0, 37.6), 'wt_plus_muse': (30.9, 37.9)}
labels = {'baseline_SAP_all_syn_TH': 'SAP all syn baseline',
          'muse': '+ MUSE EN–TH', 'wikititles': '+ WikiTitles EN–TH',
          'wt_plus_muse': '+ WT + MUSE EN–TH'}
rows = []
for name, (p1, p5) in PAPER.items():
    key = name if name == 'baseline_SAP_all_syn_TH' else run_name(name)
    file = RESULTS / key / 'result.json'
    got = json.loads(file.read_text()) if file.exists() else None
    rows.append({'variant': key, 'condition': labels[name], 'loss_params': LOSS_PARAMS,
                 'paper_P1': p1, 'paper_P5': p5,
                 'ours_P1': got['P@1'] if got else None,
                 'ours_P5': got['P@5'] if got else None,
                 'delta_P1': round(got['P@1'] - p1, 4) if got else None,
                 'delta_P5': round(got['P@5'] - p5, 4) if got else None,
                 'model_source': got.get('model_source') if got else None})
suffix = '' if LOSS_PARAMS == 'kode' else '_paperloss'
with (REPORTS / f'table3_th{suffix}.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
(REPORTS / f'table3_th{suffix}.json').write_text(json.dumps(rows, indent=2, ensure_ascii=False))
print(f'Parameter loss: {LOSS_PARAMS}')
print('Kondisi                         Paper P1/P5      Kami P1/P5')
for r in rows:
    ours = 'BELUM' if r['ours_P1'] is None else f'{r["ours_P1"]:.2f}/{r["ours_P5"]:.2f}'
    print(f'{r["condition"]:29s} {r["paper_P1"]:4.1f}/{r["paper_P5"]:4.1f}         {ours}')
print('Laporan tersimpan permanen:', REPORTS / f'table3_th{suffix}.csv')

**Analisis.**

- **Konsistensi dengan paper.** Kelima skor P@1 (Tahap 1 TH dan ZH, serta tiga varian Tahap 2) berada dalam 0,8 poin dari paper, dan semua P@5 dalam 0,6 poin. Semua selisih jauh di bawah interval kepercayaan 95% (±2,5 sampai ±2,8 poin), sehingga reproduksi dapat dianggap berhasil.
- **Urutan kondisi.** Urutan Tahap 2 sama dengan paper: baseline < MUSE < WikiTitles ≤ gabungan.
- **Model terbaik.** Untuk Thai, gabungan WikiTitles + MUSE memberi P@1 tertinggi (30,40), naik 9,3 poin dari baseline. Namun selisihnya dengan WikiTitles saja (+0,7 poin; paper +0,9) masih di dalam noise, sehingga penambahan MUSE ke WikiTitles tidak terbukti membantu, baik di paper maupun di hasil kami.
- **Ketahanan terhadap perbedaan data.** WikiTitles rekonstruksi 2026 berisi 37% lebih banyak pasangan daripada data penulis, tetapi hasilnya hanya 0,3 poin di bawah paper.
- **Kemungkinan penyebab selisih kecil.** Sampel pasangan sinonim yang berbeda (diambil acak sendiri), training per shard dengan reset state AdamW, versi data MUSE dan WikiTitles yang berbeda, serta satu kali run tanpa variasi seed.

**Ablation study.** Belum dijalankan. Dua ablasi sudah disiapkan: parameter loss versi paper (`LOSS_PARAMS = 'paper'`, hasil tersimpan terpisah dengan akhiran `_paperloss`) dan MUSE tanpa pasangan identik.

<a id="bab-11"></a>
## 11. Interpretabilitas / Error Analysis

**Celah antara P@1 dan P@5.** Pada baseline Thai, P@1 21,10 dan P@5 26,90. Artinya sekitar 58 frasa memiliki jawaban benar di lima kandidat teratas, tetapi tidak di peringkat pertama. Setelah ditambah WikiTitles + MUSE, P@5 naik ke 37,40, sehingga data terjemahan tidak hanya memperbaiki peringkat pertama, tetapi juga membawa lebih banyak jawaban benar ke kandidat teratas.

**Kapan model gagal.** Model Tahap 1 tidak pernah melihat nama berbahasa Thai di UMLS. Kemampuannya pada Thai hanya berasal dari pemahaman bahasa umum XLM-R dan sinonim bahasa lain yang ditarik ke ruang vektor yang sama. Frasa Thai yang tidak dekat dengan padanan bahasa lainnya di ruang vektor itu cenderung salah dipetakan.

**Kenapa WikiTitles lebih membantu daripada MUSE.** Faktanya, data WikiTitles jauh lebih banyak (147.100 dibanding 24.658 pasangan), berupa frasa, dan 62% pasangan MUSE identik di kedua sisi. Dugaan kami, judul artikel Wikipedia juga lebih mirip bentuk nama konsep dan frasa uji XL-BEL yang berasal dari Wikipedia.

**Belum dikerjakan.** Analisis P@1 per jenis konsep (*semantic type*), analisis kualitatif kesalahan, dan pengukuran tumpang tindih frasa uji dengan WikiTitles. Analisis kualitatif memerlukan nama kandidat dari kamus UMLS, sehingga hasilnya hanya akan disajikan secara agregat.

<a id="bab-12"></a>
## 12. Biaya Komputasi

| Proses | Hardware | Data | Waktu | Catatan |
|---|---|---|---:|---|
| Training Tahap 1 | NVIDIA L4 | 21.082.913 pasangan, 10 shard | 9,4 jam | ±2,47 iterasi/detik, ±56 menit per shard |
| Training Tahap 2, + MUSE | NVIDIA T4 | 24.658 pasangan × 5 epoch | 6,6 menit | 97 langkah per epoch |
| Training Tahap 2, + WikiTitles | NVIDIA T4 | 147.100 pasangan × 5 epoch | 32,1 menit | 575 langkah per epoch |
| Training Tahap 2, + gabungan | NVIDIA T4 | 171.758 pasangan × 5 epoch | 36,8 menit | 671 langkah per epoch |
| Evaluasi satu bahasa | NVIDIA L4 | kamus 399.931 nama + 1.000 frasa | ±5 menit | didominasi encoding kamus |

- **Ukuran model:** XLM-R base, sekitar 278 juta parameter, dengan sekitar 192 juta di antaranya berupa embedding kosakata. File `model.safetensors` berukuran 1.060,7 MB.
- **Catatan perangkat keras:** L4 hanya sedikit lebih cepat daripada T4 untuk training ini. Dugaan kami, sebagian besar waktu dipakai memperbarui matriks embedding yang besar, sementara teks inputnya pendek, sehingga pembatasnya adalah kecepatan memori, bukan daya hitung.

<a id="bab-13"></a>
## 13. Model Final / Inferensi

- **Model final Tahap 1:** XLMR + SAP_all_syn, disimpan di Drive (`sapbert_liu/runs/sapbert_full_umls2020aa/final`) dan di repositori Hugging Face privat. Hyperparameter sesuai Bab 8.1.
- **Model final Tahap 2:** tiga varian di `sapbert_liu/experiment2_th/runs/<varian>/final`. Model dengan P@1 Thai tertinggi adalah varian gabungan (30,40), tetapi selisihnya dengan WikiTitles masih dalam noise.
- **Proses inferensi:** frasa ditokenisasi (maksimal 25 token), diubah menjadi vektor [CLS], lalu dibandingkan dengan vektor semua nama di kamus. CUI dari nama terdekat menjadi prediksi.
- **Penyimpanan:** bobot model (1,06 GB) melebihi batas ukuran file GitHub (100 MB), sehingga disediakan terpisah melalui repositori Hugging Face privat. Model juga dapat dibangun ulang dengan notebook ini.

In [ ]:
from google.colab import drive
from google.colab import userdata
from huggingface_hub import HfApi
from getpass import getpass
import json

drive.mount("/content/drive")
assert "V" in globals(), "Pilih varian di Bab 6.1 terlebih dahulu."

MODEL = V["run"] / "final"

assert (MODEL / "config.json").exists(), "Model final belum ada."
assert list(MODEL.glob("*.safetensors")) or list(MODEL.glob("*.bin")), "Bobot model final belum ada."
assert (MODEL / "training_completed.json").exists(), "Training penuh belum selesai; jangan diunggah."
meta = json.loads((MODEL / "training_completed.json").read_text())
assert meta.get("variant", "all_syn") == VARIANT, "Metadata model tidak cocok dengan VARIANT (Bab 6.1)."

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if not token:
    token = getpass("Hugging Face write token (karena Secret tidak ditemukan): ")

api = HfApi(token=token)

username = api.whoami()["name"]
repo_id = f"{username}/{V['hf_repo']}"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

assert api.repo_info(repo_id).private, "Repository harus private!"

api.upload_folder(
    folder_path=str(MODEL),
    repo_id=repo_id,
    allow_patterns=[
        "*.json", "*.bin", "*.safetensors",
        "*.model", "*.txt"
    ]
)

print("Checkpoint tersedia:", repo_id)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Checkpoint tersedia: naw0510/sapbert-full-umls2020aa-private


In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel

FINAL_MODEL = STORE / 'runs/sapbert_full_umls2020aa/final'
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL)
encoder = AutoModel.from_pretrained(FINAL_MODEL).eval()

def embed(texts):
    batch = tokenizer(texts, padding=True, truncation=True, max_length=25, return_tensors='pt')
    with torch.no_grad():
        cls = encoder(**batch).last_hidden_state[:, 0]
    return torch.nn.functional.normalize(cls, dim=-1)

frasa = ['headache', 'ปวดหัว', 'sakit kepala', 'bone fracture']
vektor = embed(frasa)
pd.DataFrame((vektor @ vektor.T).numpy(), index=frasa, columns=frasa).round(3)

Contoh di atas memakai kemiripan kosinus agar nilainya mudah dibaca. Evaluasi resmi memakai *dot product* tanpa normalisasi, sesuai kode Liu.

<a id="bab-14"></a>
## 14. Pembahasan

**RQ1:** Ya. Training ulang XLM-R base dengan sinonim UMLS 2020AA semua bahasa menghasilkan P@1/P@5 21,10/26,90 untuk Thai dan 18,00/22,40 untuk Mandarin, berselisih paling besar 0,8 poin dari paper. Evaluasi checkpoint resmi dengan pipeline yang sama menghasilkan angka yang hampir identik dengan paper, sehingga selisih tersebut berasal dari training, bukan dari evaluasi.

**RQ2:** Ya. Data terjemahan umum menaikkan P@1 Thai sebesar 4,4 poin (MUSE), 8,6 poin (WikiTitles), dan 9,3 poin (gabungan), dengan urutan yang sama seperti paper. Hasil ini tetap tercapai walaupun WikiTitles harus direkonstruksi dari data 2026.

**RQ3:** Kami mencatat beberapa perbedaan antara paper dan implementasi resminya: parameter Multi-Similarity loss (Tabel 10: α = 2, β = 50, ε = 1; kode: α = 1, β = 60, base = 0,5) dan penulisan ukuran batch (512 di paper, 256 pasangan di skrip). Kami mengikuti kode resmi. Hasil kami tetap dalam noise statistik terhadap paper, tetapi pengaruh masing-masing perbedaan belum diukur secara terpisah karena ablasinya belum dijalankan.

**Insight.**
- Transfer lintas bahasa nyata: model mencapai 21% P@1 pada Thai walaupun tidak ada satu pun nama Thai di data latihnya.
- Untuk bahasa yang miskin data biomedis, data paralel umum memberi kenaikan jauh lebih besar daripada yang dapat diberikan UMLS sendiri. Bahasa Indonesia berada pada posisi serupa, sehingga temuan ini langsung relevan untuk tahap berikutnya.
- Ukuran batch memengaruhi kualitas, bukan hanya kecepatan, karena contoh negatif diambil dari dalam batch: batch 8 hanya memberi 14 pembanding negatif per nama, sedangkan batch 256 pasangan memberi 510.
- Nilai loss tidak dapat dibandingkan dengan paper. Paper tidak melaporkannya, skalanya bergantung pada parameter loss, dan miner hanya menghitung pasangan yang sulit. Loss hanya dipakai untuk memastikan training berjalan sehat.

<a id="bab-15"></a>
## 15. Keterbatasan

- Setiap konfigurasi hanya dijalankan sekali dengan seed 33, sehingga variasi antar-seed belum diketahui.
- Model Tahap 1 hasil training kami hanya dievaluasi pada Thai dan Mandarin, belum pada delapan bahasa XL-BEL lainnya.
- Varian `SAP_en_syn` dan `en_syn + MUSE` belum dilatih, walaupun datanya sudah disiapkan.
- Ablasi parameter loss dan MUSE tanpa pasangan identik belum dijalankan.
- WikiTitles adalah hasil rekonstruksi dari data 2026, dan versi file MUSE kemungkinan berbeda dari yang dipakai penulis.
- Training Tahap 1 dibagi menjadi 10 shard dengan state optimizer yang di-reset di setiap shard.
- Tumpang tindih antara frasa uji XL-BEL dan judul di WikiTitles belum diukur.
- XL-BEL banyak berisi konsep non-klinis, sehingga skornya belum mencerminkan kemampuan pada istilah klinis.
- Analisis kesalahan per jenis konsep dan analisis kualitatif belum dikerjakan.

<a id="bab-16"></a>
## 16. Kesimpulan

1. Kami mereproduksi *biomedical entity linking* lintas bahasa dari Liu dkk. (2021), dengan melatih ulang SapBERT-XLMR dari XLM-R base dan melanjutkannya dengan data terjemahan Inggris–Thai.
2. Metode utamanya adalah *self-alignment pretraining* dengan Multi-Similarity loss pada sinonim UMLS, diikuti pelatihan lanjut dengan data paralel.
3. Hasil Tahap 1 (TH 21,10/26,90; ZH 18,00/22,40) dan Tahap 2 (TH sampai 30,40/37,40) semuanya berada dalam 0,8 poin dari paper, di dalam batas noise statistik.
4. Reproduksi berhasil untuk RQ1 dan RQ2. Untuk RQ3, perbedaan antara paper dan kode resmi sudah terdokumentasi, tetapi pengaruh masing-masing belum diukur.
5. Tahap berikutnya adalah membangun data uji berbahasa Indonesia dan mengadaptasi model Tahap 1 ke bahasa Indonesia, dengan memanfaatkan temuan bahwa data tambahan dalam bahasa target memberi kenaikan terbesar.

<a id="bab-17"></a>
## 17. Referensi

[1] F. Liu, I. Vulić, A. Korhonen, dan N. Collier, "Learning Domain-Specialised Representations for Cross-Lingual Biomedical Entity Linking," dalam *Proceedings of ACL-IJCNLP 2021 (Short Papers)*, 2021.  
[2] F. Liu, E. Shareghi, Z. Meng, M. Basaldella, dan N. Collier, "Self-Alignment Pretraining for Biomedical Entity Representations," dalam *Proceedings of NAACL-HLT 2021*, 2021.  
[3] A. Conneau dkk., "Unsupervised Cross-lingual Representation Learning at Scale," dalam *Proceedings of ACL 2020*, 2020.  
[4] X. Wang, X. Han, W. Huang, D. Dong, dan M. R. Scott, "Multi-Similarity Loss with General Pair Weighting for Deep Metric Learning," dalam *Proceedings of CVPR 2019*, 2019.  
[5] A. Conneau, G. Lample, M. Ranzato, L. Denoyer, dan H. Jégou, "Word Translation Without Parallel Data," dalam *Proceedings of ICLR 2018*, 2018.  
[6] A. M. Ningtyas dkk., "AnnoHID: LLM-Assisted Annotation Framework for Low-Resource Medical Texts," dalam *Proceedings of ACL 2026 (System Demonstrations)*, 2026. https://aclanthology.org/2026.acl-demo.67  
[7] O. Bodenreider, "The Unified Medical Language System (UMLS): integrating biomedical terminology," *Nucleic Acids Research*, vol. 32, hlm. D267–D270, 2004.  
[8] K. Musgrave, S. Belongie, dan S.-N. Lim, "PyTorch Metric Learning," arXiv:2008.09164, 2020.  
[9] Repo kode `cambridgeltl/sapbert`, GitHub, commit `1a528f6`.  
[10] Dataset `wikimedia/wikidata-title-desc`, Hugging Face, revisi `snapshot-2026-03-30`.